In [1]:
import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter
import matplotlib.pyplot as plt


# ------------------------------------------------------------------------------
# 0) CONFIG (best-practice for clarity + minimal heuristics)
# ------------------------------------------------------------------------------
CONFIG = {
    "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
    "target_activities": [6, 7, 12],
    "fs": 50,
    "window_size": 100,
    "stride": 50,
    "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

    "batch_size": 64,
    "epochs": 50,
    "lr": 1e-3,
    "seed": 42,
    "latent_dim": 64,
    "hidden_dim": 128,

    # losses
    "lambda_recon": 1.0,
    "lambda_bce": 0.5,      # confident hard labels only
    "lambda_pair": 0.3,     # SS/TT/ST/TS separation
    "lambda_inertial": 0.2,

    # confident labeling (NO threshold tuning, just quantile partitions)
    "q_conf": 0.20,         # top/bottom q used as confident; middle ignored

    # detector-only smoothing for gravity estimation (MODEL INPUT stays raw-normalized)
    "detector_smooth": True,
    "smooth_method": "savgol",  # "savgol" or "ema"
    "savgol_win": 11,           # odd
    "savgol_poly": 2,
    "ema_alpha": 0.15,

    # segmentize only for visualization (derived from confident mask, not used as supervision)
    "seg_min_len_sec": 0.20,
    "seg_max_gap_sec": 0.10,

    "out_dir": "./out_main_engine_best",
    "plot_sec": 60,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["out_dir"], exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])


# ------------------------------------------------------------------------------
# 1) MHEALTH LOADING
# ------------------------------------------------------------------------------
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count.")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id
        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(
        f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}"
    )
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    # MHEALTH columns: based on common mapping used in your code
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# ------------------------------------------------------------------------------
# 2) Physics-based Transition Score (minimal + interpretable)
#    r(t) = a * ||omega(t)|| + b * ||Δ g_hat(t)|| + c * Var_W(g_hat(t))
# ------------------------------------------------------------------------------
def robust_zscore(x: np.ndarray, eps=1e-6, mad_floor=1e-3):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    mad = max(float(mad), float(mad_floor))
    return (x - med) / mad


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def ema_smooth(x: np.ndarray, alpha: float):
    if x.ndim == 1:
        y = np.zeros_like(x, dtype=np.float32)
        y[0] = x[0]
        for t in range(1, len(x)):
            y[t] = alpha * x[t] + (1 - alpha) * y[t - 1]
        return y.astype(np.float32)
    else:
        y = np.zeros_like(x, dtype=np.float32)
        y[0, :] = x[0, :]
        for t in range(1, x.shape[0]):
            y[t, :] = alpha * x[t, :] + (1 - alpha) * y[t - 1, :]
        return y.astype(np.float32)


def savgol_smooth(x: np.ndarray, win: int, poly: int):
    win = int(win)
    if win < 3:
        return x.astype(np.float32)
    if win % 2 == 0:
        win += 1
    if x.ndim == 1:
        if len(x) < win:
            return x.astype(np.float32)
        return savgol_filter(x, window_length=win, polyorder=poly).astype(np.float32)
    else:
        T, C = x.shape
        if T < win:
            return x.astype(np.float32)
        y = np.zeros_like(x, dtype=np.float32)
        for c in range(C):
            y[:, c] = savgol_filter(x[:, c], window_length=win, polyorder=poly).astype(np.float32)
        return y


def maybe_smooth_for_detector(Xg: np.ndarray, cfg):
    if Xg is None:
        return None
    if not cfg.get("detector_smooth", False):
        return Xg.astype(np.float32)
    method = cfg.get("smooth_method", "savgol")
    if method == "ema":
        return ema_smooth(Xg.astype(np.float32), alpha=float(cfg.get("ema_alpha", 0.15)))
    return savgol_smooth(
        Xg.astype(np.float32),
        win=int(cfg.get("savgol_win", 11)),
        poly=int(cfg.get("savgol_poly", 2)),
    )


def unit_vec(x: np.ndarray, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


def gravity_dir_from_acc(acc3: np.ndarray, cfg):
    """
    Estimate gravity direction by low-pass smoothing acc then normalizing.
    (This is physics, not heuristic feature engineering.)
    """
    acc_lp = maybe_smooth_for_detector(acc3, cfg)
    ghat = unit_vec(acc_lp)
    return ghat.astype(np.float32)


def dir_change_score(ghat: np.ndarray):
    """
    ||Δ ghat|| approx via 1 - cos similarity
    """
    T = len(ghat)
    cos_prev = np.sum(ghat[1:] * ghat[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)
    d = np.zeros(T, dtype=np.float32)
    d[1:] = 1.0 - cos_prev
    return d


def dir_var_score(ghat: np.ndarray, var_win: int = 10):
    """
    window variance of ghat (how unstable gravity direction is)
    """
    T = len(ghat)
    half = var_win // 2
    v = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = ghat[s:e]
        v[i] = float(np.mean(np.var(local, axis=0)))
    return v.astype(np.float32)


def gyro_mag_score(gyro3: np.ndarray, cfg):
    """
    magnitude of angular velocity (optionally smoothed for detector stability)
    """
    g = maybe_smooth_for_detector(gyro3, cfg)
    mag = np.linalg.norm(g, axis=1).astype(np.float32)
    return mag


def physics_transition_score(raw_24: np.ndarray, cfg):
    """
    Build a single r(t) from available acc/gyro groups.
    Steps:
      - For each acc sensor: ghat -> d_dir + var_dir
      - For each gyro sensor: |omega|
      - robust z-score each component and average (minimal fusion)
    """
    comps = []

    # ACC-based components (gravity direction change)
    for gname in ["chest_acc", "ankle_acc", "arm_acc"]:
        acc = get_group_array_from_block(raw_24, gname)
        if acc is None:
            continue
        ghat = gravity_dir_from_acc(acc, cfg)
        d = dir_change_score(ghat)
        v = dir_var_score(ghat, var_win=10)
        comps.append(robust_zscore(d))
        comps.append(robust_zscore(v))

    # GYRO-based components (angular velocity)
    for gname in ["ankle_gyro", "arm_gyro"]:
        gyro = get_group_array_from_block(raw_24, gname)
        if gyro is None:
            continue
        w = gyro_mag_score(gyro, cfg)
        comps.append(robust_zscore(w))

    if len(comps) == 0:
        raise ValueError("No available sensors to compute physics score.")

    R = np.stack(comps, axis=0).mean(axis=0).astype(np.float32)  # average fusion
    return R  # unbounded-ish (z-score scale)


def make_confident_labels_from_score(R: np.ndarray, q_conf: float):
    """
    R: transition score (higher => more transition)
    returns:
      y_hard: {0,1} (float)
      m_conf: {0,1} mask for confident points
      y_soft: sigmoid-scaled version (for plotting/weights only)
      s_soft: 1 - y_soft
      thr_lo, thr_hi
    """
    q = float(np.clip(q_conf, 0.01, 0.49))
    thr_lo = float(np.quantile(R, q))
    thr_hi = float(np.quantile(R, 1.0 - q))

    # hard labels only for extremes
    y_hard = np.zeros_like(R, dtype=np.float32)
    m_conf = np.zeros_like(R, dtype=np.float32)

    m_lo = (R <= thr_lo)
    m_hi = (R >= thr_hi)

    y_hard[m_hi] = 1.0
    y_hard[m_lo] = 0.0
    m_conf[m_hi] = 1.0
    m_conf[m_lo] = 1.0

    # soft for weighting/visualization (NOT treated as calibrated probability)
    y_soft = sigmoid_np(R / 1.0).astype(np.float32)  # scale 1.0 = mild smoothing of logits
    s_soft = (1.0 - y_soft).astype(np.float32)

    return y_hard, m_conf, y_soft, s_soft, thr_lo, thr_hi


# ------------------------------------------------------------------------------
# 3) Segmentize for visualization (based on confident transition mask only)
# ------------------------------------------------------------------------------
def mask_fill_gaps(mask: np.ndarray, max_gap: int):
    if max_gap <= 0:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if m[i]:
            i += 1
            continue
        j = i
        while j < T and (not m[j]):
            j += 1
        gap_len = j - i
        if i > 0 and j < T and gap_len <= max_gap:
            m[i:j] = True
        i = j
    return m


def mask_remove_short_islands(mask: np.ndarray, min_len: int):
    if min_len <= 1:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        seg_len = j - i
        if seg_len < min_len:
            m[i:j] = False
        i = j
    return m


def segments_from_transition_mask(mask_tr: np.ndarray, cfg):
    fs = int(cfg["fs"])
    min_len = int(round(float(cfg.get("seg_min_len_sec", 0.2)) * fs))
    max_gap = int(round(float(cfg.get("seg_max_gap_sec", 0.1)) * fs))

    m = mask_tr.astype(bool)
    m = mask_fill_gaps(m, max_gap=max_gap)
    m = mask_remove_short_islands(m, min_len=min_len)

    T = len(m)
    segs = []
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        segs.append((i, j))
        i = j
    return m, segs


# ------------------------------------------------------------------------------
# 4) Dataset (physics score computed per block -> window slices)
# ------------------------------------------------------------------------------
class MainEngineDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]
        self.samples = []

        for b in blocks:
            raw = b["raw"]
            subject = b["subject"]
            act = b["act"]
            T = len(raw)

            # model input groups (RAW)
            full_groups_raw = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups_raw[g] = None if Xg is None else Xg.astype(np.float32)

            # physics transition score r(t)
            R = physics_transition_score(raw, cfg)  # (T,)
            y_hard, m_conf, y_soft, s_soft, thr_lo, thr_hi = make_confident_labels_from_score(
                R, q_conf=cfg["q_conf"]
            )

            avail_groups = [g for g in self.groups if full_groups_raw.get(g, None) is not None]
            if len(avail_groups) == 0:
                continue

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win

                # concat sensors for model input
                x_parts = [full_groups_raw[g][st:ed] for g in avail_groups]
                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)

                # per-window normalization (kept same as your flow)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append(
                    {
                        "x": Xcat,                           # (win,C)
                        "s_soft": s_soft[st:ed].astype(np.float32),   # (win,) soft steadiness weights
                        "y_hard": y_hard[st:ed].astype(np.float32),   # (win,) 0/1 (only valid where mask=1)
                        "m_conf": m_conf[st:ed].astype(np.float32),   # (win,) 1 for confident, else 0
                        "y_soft": y_soft[st:ed].astype(np.float32),   # (win,) for plotting only
                        "meta": {
                            "subject": subject, "act": act,
                            "thr_lo": thr_lo, "thr_hi": thr_hi
                        },
                    }
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s_soft"], dtype=torch.float32)             # (T,)
        y = torch.tensor(d["y_hard"], dtype=torch.float32)             # (T,)
        m = torch.tensor(d["m_conf"], dtype=torch.float32)             # (T,)
        ysoft = torch.tensor(d["y_soft"], dtype=torch.float32)         # (T,)
        return x, s, y, m, ysoft


# ------------------------------------------------------------------------------
# 5) Model
# ------------------------------------------------------------------------------
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )
        self.trans_head = nn.Sequential(
            nn.Conv1d(latent_dim, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=1),
        )
        self.grav_proj = nn.Linear(latent_dim, 3)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.to_latent(h)                  # (B,D,T)
        p = torch.sigmoid(self.trans_head(z))  # (B,1,T)
        x_recon = self.decoder(z)              # (B,C,T)
        return z, p, x_recon


# ------------------------------------------------------------------------------
# 6) Losses
# ------------------------------------------------------------------------------
def masked_bce(p_hat, y_hard, m_conf, eps=1e-6):
    """
    BCE only on confident points (m_conf == 1)
    """
    p = p_hat.squeeze(1).clamp(eps, 1 - eps)  # (B,T)
    y = y_hard.clamp(0, 1)
    m = m_conf

    denom = (m.sum() + 1e-6)
    loss = (F.binary_cross_entropy(p, y, reduction="none") * m).sum() / denom
    return loss


def pair_consistency_loss(z, s_soft, delta: int = 5, margin: float = 0.2):
    """
    Uses soft steadiness s(t) as weights (no hard thresholds).
    Your SS/TT/ST/TS separation intuition remains.
    """
    B, D, T = z.shape
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s_soft[:, :-delta]
    s2 = s_soft[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)  # (B,T-d)

    # high s -> steady, low s -> transition
    w_pos = (s1 * s2).detach()
    w_neg = ((1 - s1) * (1 - s2)).detach()

    pos_loss = (w_pos * (1.0 - sim)).mean()
    neg_loss = (w_neg * F.relu(sim - margin)).mean()
    return pos_loss + neg_loss


def inertial_invariance_loss(z, x, s_soft, model, eps=1e-6):
    """
    Keep your inertial loss form: encourage gravity-related projection to correlate with avg gravity.
    """
    B, D, T = z.shape
    C = x.shape[1]
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, : nvec * 3, :].reshape(B, nvec, 3, T)
    g = x_reshaped.mean(dim=3).mean(dim=1)  # (B,3)
    g = F.normalize(g, dim=1)

    w = s_soft / (s_soft.sum(dim=1, keepdim=True) + eps)  # emphasize steady
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)  # (B,D)

    z3 = model.grav_proj(z_bar)
    z3 = F.normalize(z3, dim=1)
    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# ------------------------------------------------------------------------------
# 7) Train / Eval
# ------------------------------------------------------------------------------
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for _ in range(cfg["epochs"]):
        for x, s_soft, y_hard, m_conf, _ in loader:
            x = x.to(DEVICE)
            s_soft = s_soft.to(DEVICE)
            y_hard = y_hard.to(DEVICE)
            m_conf = m_conf.to(DEVICE)

            z, p, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_bce = masked_bce(p, y_hard, m_conf)
            loss_pair = pair_consistency_loss(z, s_soft, delta=5, margin=0.2)
            loss_inert = inertial_invariance_loss(z, x, s_soft, model)

            loss = (
                cfg["lambda_recon"] * loss_recon
                + cfg["lambda_bce"] * loss_bce
                + cfg["lambda_pair"] * loss_pair
                + cfg["lambda_inertial"] * loss_inert
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()


@torch.no_grad()
def eval_conf_bce(model, loader):
    """
    Evaluate BCE on confident points only (more meaningful than MAE vs soft y)
    """
    model.eval()
    total, denom = 0.0, 0.0
    for x, _, y_hard, m_conf, _ in loader:
        x = x.to(DEVICE)
        y_hard = y_hard.to(DEVICE)
        m_conf = m_conf.to(DEVICE)

        _, p, _ = model(x)
        p = p.squeeze(1).clamp(1e-6, 1 - 1e-6)
        bce = F.binary_cross_entropy(p, y_hard, reduction="none")
        total += float((bce * m_conf).sum().item())
        denom += float(m_conf.sum().item())
    return total / (denom + 1e-6)


# ------------------------------------------------------------------------------
# 8) Plot helpers (for sanity)
# ------------------------------------------------------------------------------
def plot_raw_and_scores(raw_24, y_soft, p_hat, segs, thr_hi_score, cfg, title, save_path):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(raw_24))
    t = np.arange(T_show) / fs

    def mag(g):
        Xg = get_group_array_from_block(raw_24[:T_show], g)
        return None if Xg is None else np.linalg.norm(Xg, axis=1)

    chest_acc = mag("chest_acc")
    ankle_acc = mag("ankle_acc")
    arm_acc = mag("arm_acc")
    ankle_gyro = mag("ankle_gyro")
    arm_gyro = mag("arm_gyro")

    plt.figure(figsize=(18, 7))
    ax1 = plt.subplot(2, 1, 1)

    for (s, e) in segs:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax1.axvspan(ss / fs, ee / fs, alpha=0.12)

    if chest_acc is not None: ax1.plot(t, chest_acc, label="chest_acc |mag|", alpha=0.9)
    if ankle_acc is not None: ax1.plot(t, ankle_acc, label="ankle_acc |mag|", alpha=0.9)
    if arm_acc is not None: ax1.plot(t, arm_acc, label="arm_acc |mag|", alpha=0.9)
    if ankle_gyro is not None: ax1.plot(t, ankle_gyro, label="ankle_gyro |mag|", alpha=0.7)
    if arm_gyro is not None: ax1.plot(t, arm_gyro, label="arm_gyro |mag|", alpha=0.7)

    ax1.set_title(title + " | raw magnitudes (shaded = confident transition segments)")
    ax1.grid(alpha=0.25)
    ax1.legend(ncol=3)

    ax2 = plt.subplot(2, 1, 2)
    for (s, e) in segs:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax2.axvspan(ss / fs, ee / fs, alpha=0.12)

    ax2.plot(t, y_soft[:T_show], label="physics y_soft (sigmoid(score))", alpha=0.8)
    if p_hat is not None:
        ax2.plot(t, p_hat[:T_show], label="model p_hat", alpha=0.9)

    ax2.axhline(thr_hi_score, linestyle="--", linewidth=1.2, label=f"conf trans thr (top-q, q={cfg['q_conf']:.2f})")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(alpha=0.25)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


@torch.no_grad()
def infer_p_hat_on_block(model, raw_24: np.ndarray, cfg):
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    # build raw groups
    full_groups = {}
    for g in cfg["groups"]:
        Xg = get_group_array_from_block(raw_24, g)
        full_groups[g] = None if Xg is None else Xg.astype(np.float32)

    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    p_sum = np.zeros(T, dtype=np.float32)
    p_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)
        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        _, p, _ = model(x)
        pw = p.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)

        p_sum[st:ed] += pw
        p_cnt[st:ed] += 1.0

    return p_sum / (p_cnt + 1e-8)


def pick_representative_fold(df_act: pd.DataFrame):
    if len(df_act) == 0:
        return None
    med = df_act["bce_conf"].median()
    idx = (df_act["bce_conf"] - med).abs().idxmin()
    return df_act.loc[idx].to_dict()


# ------------------------------------------------------------------------------
# 9) Main
# ------------------------------------------------------------------------------
def main():
    print("=" * 70)
    print("Main Engine (BEST, physics-defined pseudo labels + confident-only supervision)")
    print("=" * 70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, CONFIG["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    rep_artifacts = {}

    for act in acts:
        print("\n" + "=" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("=" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = MainEngineDataset(train_blocks, CONFIG)
            test_ds = MainEngineDataset(test_blocks, CONFIG)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

            x0, _, _, _, _ = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
            ).to(DEVICE)

            train_one_fold(model, train_loader, CONFIG)
            bce_conf = eval_conf_bce(model, test_loader)

            records.append(
                {
                    "act": act,
                    "test_sub": test_sub,
                    "bce_conf": bce_conf,
                    "input_ch": input_ch,
                    "q_conf": CONFIG["q_conf"],
                    "detector_smooth": int(CONFIG["detector_smooth"]),
                    "smooth_method": CONFIG["smooth_method"],
                }
            )

            print(f"  Fold test_sub={test_sub:2d} | BCE(conf)={bce_conf:.4f}")

            # store representative artifacts
            b0 = test_blocks[0]
            rep_artifacts.setdefault(act, [])
            rep_artifacts[act].append(
                {
                    "test_sub": test_sub,
                    "bce_conf": bce_conf,
                    "raw0": b0["raw"],
                    "subject": b0["subject"],
                    "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "input_ch": input_ch,
                }
            )

    df_res = pd.DataFrame(records)
    csv_path = os.path.join(CONFIG["out_dir"], "results_loso_best.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # representative plots per activity
    for act in acts:
        df_act = df_res[df_res["act"] == act].reset_index(drop=True)
        if len(df_act) == 0 or act not in rep_artifacts:
            continue

        rep = pick_representative_fold(df_act)
        if rep is None:
            continue
        rep_sub = int(rep["test_sub"])

        cand = None
        for item in rep_artifacts[act]:
            if int(item["test_sub"]) == rep_sub:
                cand = item
                break
        if cand is None:
            continue

        model = MainEngineNet(
            input_ch=int(cand["input_ch"]),
            hidden_dim=CONFIG["hidden_dim"],
            latent_dim=CONFIG["latent_dim"],
        ).to(DEVICE)
        model.load_state_dict(cand["model_state"], strict=True)
        model.eval()

        raw0 = cand["raw0"]
        subj = cand["subject"]

        # physics score & confident masks (for segment shading)
        R = physics_transition_score(raw0, CONFIG)
        y_hard, m_conf, y_soft, s_soft, thr_lo, thr_hi = make_confident_labels_from_score(R, CONFIG["q_conf"])
        mask_tr = (m_conf > 0.5) & (y_hard > 0.5)  # confident transition points
        mask_tr, segs = segments_from_transition_mask(mask_tr, CONFIG)

        p_blk = infer_p_hat_on_block(model, raw0, CONFIG)

        # draw threshold line in y_soft scale for visualization
        thr_hi_ysoft = float(sigmoid_np(thr_hi / 1.0))

        p1_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_raw_ysoft_pHat_sub{subj}.png")
        plot_raw_and_scores(
            raw0, y_soft, p_blk, segs, thr_hi_ysoft, CONFIG,
            f"Act{act} RepFold (test_sub={rep_sub})",
            p1_path
        )
        print(f"[Saved plot] act={act} rep_sub={rep_sub} -> {p1_path}")

    print("\nDone.")


if __name__ == "__main__":
    main()


Main Engine (BEST, physics-defined pseudo labels + confident-only supervision)
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | BCE(conf)=0.2525
  Fold test_sub= 2 | BCE(conf)=1.0888
  Fold test_sub= 3 | BCE(conf)=0.7356
  Fold test_sub= 4 | BCE(conf)=1.1212
  Fold test_sub= 5 | BCE(conf)=0.5572
  Fold test_sub= 6 | BCE(conf)=0.7148
  Fold test_sub= 7 | BCE(conf)=1.0341
  Fold test_sub= 8 | BCE(conf)=0.4572
  Fold test_sub= 9 | BCE(conf)=0.3792
  Fold test_sub=10 | BCE(conf)=0.6163

[Activity 7] Single-activity LOSO
[Blocks] act=7 total=10 | folds=10
  Fold test_sub= 1 | BCE(conf)=0.1974
  Fold test_sub= 2 | BCE(conf)=0.6193
  Fold test_sub= 3 | BCE(conf)=0.4586
  Fold test_sub= 4 | BCE(conf)=0.2513
  Fold test_sub= 5 | BCE(con

In [2]:
# ============================================================
# Main Engine (BEST v2)
# - physics-defined pseudo labels (direction/axis change, not magnitude)
#   * ACC  : gravity direction change (LPF on acc -> g_hat -> change + local var)
#   * GYRO : gyro axis direction change (gyro_hat -> change + local var)
# - detector-only smoothing (MODEL INPUT is RAW)
# - fusion by self-consistency quality q (softmax weights)
# - confident-only supervision (BCE on top-q transition + bottom-q steady)
# - pair separation analysis (SS/TT/ST/TS) per-fold + aggregated per-activity plots
# ============================================================

import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------------
CONFIG = {
    "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
    "target_activities": [6, 7, 12],
    "fs": 50,
    "window_size": 100,
    "stride": 50,
    "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

    "batch_size": 64,
    "epochs": 50,
    "lr": 1e-3,
    "seed": 42,

    "latent_dim": 64,
    "hidden_dim": 128,

    # losses
    "lambda_recon": 1.0,
    "lambda_conf": 0.7,     # confident-only BCE
    "lambda_pair": 0.3,
    "lambda_inertial": 0.2,

    # --- pseudo label calibration in [0,1] ---
    # robust quantile rescale to avoid saturation
    "calib_q": 0.15,

    # --- confident supervision quantile (both sides) ---
    # transition: top-q ; steady: bottom-q
    "conf_q": 0.20,

    # --- detector-only smoothing (MODEL INPUT RAW) ---
    "detector_smooth": True,
    "smooth_method": "savgol",  # "savgol" or "ema"
    "savgol_win": 21,           # odd; slightly stronger LPF for gravity extraction
    "savgol_poly": 2,
    "ema_alpha": 0.12,

    # --- direction change score ---
    "var_win": 11,              # local variance window (odd recommended)

    # --- segmentize for visualization ---
    "seg_q": 0.20,
    "seg_min_len_sec": 0.20,
    "seg_max_gap_sec": 0.10,

    "out_dir": "./out_main_engine_best_v2",
    "plot_sec": 60,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["out_dir"], exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])


# ------------------------------------------------------------------
# 1) MHEALTH LOADING
# ------------------------------------------------------------------
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        try:
            df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        except:
            df = pd.read_csv(fpath, sep="\t", header=None)

        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count (expected >= 24).")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id
        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(
        f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}"
    )
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    # MHEALTH columns mapping (as in your code)
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# ------------------------------------------------------------------
# 2) Detector (direction-change) + Fusion
# ------------------------------------------------------------------
def robust_zscore(x: np.ndarray, eps=1e-6, mad_floor=1e-3):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    mad = max(float(mad), float(mad_floor))
    return (x - med) / mad


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def ema_smooth(x: np.ndarray, alpha: float):
    if x.ndim == 1:
        y = np.zeros_like(x, dtype=np.float32)
        y[0] = x[0]
        for t in range(1, len(x)):
            y[t] = alpha * x[t] + (1 - alpha) * y[t - 1]
        return y.astype(np.float32)
    else:
        y = np.zeros_like(x, dtype=np.float32)
        y[0, :] = x[0, :]
        for t in range(1, x.shape[0]):
            y[t, :] = alpha * x[t, :] + (1 - alpha) * y[t - 1, :]
        return y.astype(np.float32)


def savgol_smooth(x: np.ndarray, win: int, poly: int):
    win = int(win)
    if win < 3:
        return x.astype(np.float32)
    if win % 2 == 0:
        win += 1
    if x.ndim == 1:
        if len(x) < win:
            return x.astype(np.float32)
        return savgol_filter(x, window_length=win, polyorder=poly).astype(np.float32)
    else:
        T, C = x.shape
        if T < win:
            return x.astype(np.float32)
        y = np.zeros_like(x, dtype=np.float32)
        for c in range(C):
            y[:, c] = savgol_filter(x[:, c], window_length=win, polyorder=poly).astype(np.float32)
        return y


def maybe_smooth_for_detector(Xg: np.ndarray, cfg):
    if Xg is None:
        return None
    if not cfg.get("detector_smooth", False):
        return Xg.astype(np.float32)
    method = cfg.get("smooth_method", "savgol")
    if method == "ema":
        return ema_smooth(Xg.astype(np.float32), alpha=float(cfg.get("ema_alpha", 0.12)))
    return savgol_smooth(
        Xg.astype(np.float32),
        win=int(cfg.get("savgol_win", 21)),
        poly=int(cfg.get("savgol_poly", 2)),
    )


def unit_vec(X: np.ndarray, eps=1e-8):
    n = np.linalg.norm(X, axis=1, keepdims=True) + eps
    return (X / n).astype(np.float32)


def local_unit_var(U: np.ndarray, var_win: int):
    # U: (T,3) unit vector
    T = len(U)
    if var_win < 3:
        var_win = 3
    if var_win % 2 == 0:
        var_win += 1
    half = var_win // 2
    v = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = U[s:e]
        v[i] = float(np.mean(np.var(local, axis=0)))
    return v


def dir_change(U: np.ndarray):
    # U: (T,3) unit vectors
    T = len(U)
    c = np.zeros(T, dtype=np.float32)
    if T <= 1:
        return c
    cos_prev = np.sum(U[1:] * U[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)
    c[1:] = 1.0 - cos_prev
    return c.astype(np.float32)


def gravity_direction_change_score(acc3_det: np.ndarray, cfg):
    # detector-only LPF already applied to acc3_det
    U = unit_vec(acc3_det)
    c = dir_change(U)
    v = local_unit_var(U, int(cfg.get("var_win", 11)))
    return (c + v).astype(np.float32)


def gyro_axis_change_score(gyro3_det: np.ndarray, cfg):
    # direction of angular velocity vector (axis of rotation)
    U = unit_vec(gyro3_det)
    c = dir_change(U)
    v = local_unit_var(U, int(cfg.get("var_win", 11)))
    return (c + v).astype(np.float32)


def compute_group_change(group_key: str, Xg_det: np.ndarray, cfg):
    if Xg_det is None:
        return None
    if "acc" in group_key:
        return gravity_direction_change_score(Xg_det, cfg)
    if "gyro" in group_key:
        return gyro_axis_change_score(Xg_det, cfg)
    # fallback
    return np.linalg.norm(np.diff(Xg_det, axis=0, prepend=Xg_det[:1]), axis=1).astype(np.float32)


def steady_score_from_change(c: np.ndarray):
    # change score c: larger => more transition-like
    cz = robust_zscore(c, eps=1e-6, mad_floor=1e-3)
    b = np.percentile(cz, 80)                 # relative reference inside-trial (threshold-free)
    y = sigmoid_np((cz - b) / 0.6)            # transition prob (soft)
    s = 1.0 - y                               # steady score
    if len(s) >= 11:
        s = savgol_filter(s, window_length=11, polyorder=2).astype(np.float32)
        s = np.clip(s, 0.0, 1.0)
    return s.astype(np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray, eps=1e-8):
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    return float(np.dot(a, b) / (na * nb))


def estimate_self_consistency_quality(group_key: str, X_raw: np.ndarray, s_full: np.ndarray):
    # quality = similarity(high-steady) - similarity(low-steady)
    q = 0.2
    k_pairs = 128

    T = len(s_full)
    order = np.argsort(s_full)
    n = max(5, int(q * T))
    low_idx = order[:n]
    high_idx = order[-n:]
    if len(low_idx) < 5 or len(high_idx) < 5:
        return 0.0

    # use RAW, normalized direction for both acc/gyro to measure self-consistency
    V = X_raw / (np.linalg.norm(X_raw, axis=1, keepdims=True) + 1e-8)

    def mean_pair_sim(idxs):
        sims = []
        for _ in range(k_pairs):
            i, j = np.random.choice(idxs, size=2, replace=True)
            sims.append(cosine_sim(V[i], V[j]))
        return float(np.mean(sims))

    S_high = mean_pair_sim(high_idx)
    S_low = mean_pair_sim(low_idx)
    return float(S_high - S_low)


def fuse_group_scores_block(groups_dict):
    avail = [(g, d) for g, d in groups_dict.items() if d["s_full"] is not None]
    if not avail:
        raise ValueError("No available groups for fusion.")

    qs = np.array([d["q_full"] for _, d in avail], dtype=np.float32)
    tau = 0.25
    ws = np.exp(qs / max(tau, 1e-6))
    ws = ws / (ws.sum() + 1e-8)

    T = len(avail[0][1]["s_full"])
    s_fused = np.zeros(T, dtype=np.float32)
    weights, qualities = {}, {}
    for (g, d), w in zip(avail, ws):
        s_fused += float(w) * d["s_full"]
        weights[g] = float(w)
        qualities[g] = float(d["q_full"])

    s_fused = np.clip(s_fused, 0.0, 1.0)
    return s_fused, weights, qualities


def quantile_calibrate_01(y: np.ndarray, q: float, eps: float = 1e-6):
    # symmetric quantile stretch to reduce saturation
    q = float(np.clip(q, 0.0, 0.49))
    lo = float(np.quantile(y, q))
    hi = float(np.quantile(y, 1.0 - q))
    denom = max(hi - lo, eps)
    y_cal = (y - lo) / denom
    return np.clip(y_cal, 0.0, 1.0).astype(np.float32)


# ---- segmentize for shading (top-q + run-length filtering)
def mask_fill_gaps(mask: np.ndarray, max_gap: int):
    if max_gap <= 0:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if m[i]:
            i += 1
            continue
        j = i
        while j < T and (not m[j]):
            j += 1
        gap_len = j - i
        if i > 0 and j < T and gap_len <= max_gap:
            m[i:j] = True
        i = j
    return m


def mask_remove_short_islands(mask: np.ndarray, min_len: int):
    if min_len <= 1:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        seg_len = j - i
        if seg_len < min_len:
            m[i:j] = False
        i = j
    return m


def segmentize_from_y(y01: np.ndarray, cfg):
    q = float(cfg.get("seg_q", 0.2))
    thr_hi = float(np.quantile(y01, 1.0 - q))

    mask = (y01 >= thr_hi)

    fs = int(cfg["fs"])
    min_len = int(round(float(cfg.get("seg_min_len_sec", 0.2)) * fs))
    max_gap = int(round(float(cfg.get("seg_max_gap_sec", 0.1)) * fs))

    mask = mask_fill_gaps(mask, max_gap=max_gap)
    mask = mask_remove_short_islands(mask, min_len=min_len)

    m = mask.astype(bool)
    T = len(m)
    segments = []
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        segments.append((i, j))
        i = j

    return m, segments, thr_hi


# ------------------------------------------------------------------
# 3) Dataset (block-level detector->fusion -> window slice)
# ------------------------------------------------------------------
class MainEngineDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]
        self.samples = []  # {"x":(win,C),"s":(win,),"y":(win,), "meta":{...}}

        for b in blocks:
            raw = b["raw"]
            subject = b["subject"]
            act = b["act"]
            T = len(raw)

            # RAW groups for MODEL INPUT (no smoothing)
            full_groups_raw = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups_raw[g] = None if Xg is None else Xg.astype(np.float32)

            # detector groups (smoothed only for detector)
            groups_dict = {}
            for g in self.groups:
                X_raw = full_groups_raw.get(g, None)
                if X_raw is None:
                    continue

                X_det = maybe_smooth_for_detector(X_raw, cfg)
                c_full = compute_group_change(g, X_det, cfg)
                s_full = steady_score_from_change(c_full)

                # quality computed on RAW direction-consistency (avoid "cheating" via smoothing)
                q_full = estimate_self_consistency_quality(g, X_raw, s_full)
                groups_dict[g] = {"X_full": X_raw, "s_full": s_full, "q_full": float(q_full)}

            if len(groups_dict) == 0:
                continue

            s_fused_full, w_block, q_block = fuse_group_scores_block(groups_dict)
            y_full = (1.0 - s_fused_full).astype(np.float32)  # transition score
            y_full = quantile_calibrate_01(y_full, q=cfg["calib_q"])
            s_fused_full = (1.0 - y_full).astype(np.float32)

            avail_groups = [g for g in self.groups if full_groups_raw.get(g, None) is not None]

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win
                x_parts = [full_groups_raw[g][st:ed] for g in avail_groups]
                if len(x_parts) == 0:
                    continue

                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append(
                    {
                        "x": Xcat,
                        "s": s_fused_full[st:ed].astype(np.float32),
                        "y": y_full[st:ed].astype(np.float32),
                        "meta": {"subject": subject, "act": act, "weights": w_block, "qualities": q_block},
                    }
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s"], dtype=torch.float32)                  # (T,)
        y = torch.tensor(d["y"], dtype=torch.float32)                  # (T,)
        return x, s, y


# ------------------------------------------------------------------
# 4) Model
# ------------------------------------------------------------------
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )
        self.trans_head = nn.Sequential(
            nn.Conv1d(latent_dim, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=1),
        )
        self.grav_proj = nn.Linear(latent_dim, 3)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.to_latent(h)                  # (B,D,T)
        p = torch.sigmoid(self.trans_head(z))  # (B,1,T)
        x_recon = self.decoder(z)              # (B,C,T)
        return z, p, x_recon


# ------------------------------------------------------------------
# 5) Losses
# ------------------------------------------------------------------
def confident_bce(p_hat, y_soft, conf_q: float, eps=1e-6):
    """
    Confident-only BCE using BOTH tails:
      - y in top-q  -> target 1 (transition)
      - y in bottom-q -> target 0 (steady)
    This makes supervision much sharper while staying threshold-free (quantile-based).
    """
    p = p_hat.squeeze(1)  # (B,T)
    y = y_soft            # (B,T)

    # compute thresholds per-batch
    y_np = y.detach().cpu().numpy()
    thr_hi = np.quantile(y_np, 1.0 - conf_q)
    thr_lo = np.quantile(y_np, conf_q)

    mask_hi = (y >= float(thr_hi))
    mask_lo = (y <= float(thr_lo))
    mask = (mask_hi | mask_lo)

    if mask.sum().item() == 0:
        return torch.tensor(0.0, device=p.device), 0.0, float(thr_lo), float(thr_hi)

    target = torch.zeros_like(y)
    target[mask_hi] = 1.0
    target[mask_lo] = 0.0

    loss = F.binary_cross_entropy(p[mask].clamp(eps, 1 - eps), target[mask].clamp(eps, 1 - eps))
    ratio = float(mask.float().mean().item())
    return loss, ratio, float(thr_lo), float(thr_hi)


def pair_consistency_loss(z, s, delta: int = 5, margin: float = 0.2):
    B, D, T = z.shape
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s[:, :-delta]
    s2 = s[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)  # (B,T-d)

    w_pos = (s1 * s2).detach()
    w_neg = ((1 - s1) * (1 - s2)).detach()

    pos_loss = (w_pos * (1.0 - sim)).mean()
    neg_loss = (w_neg * F.relu(sim - margin)).mean()
    return pos_loss + neg_loss


def inertial_invariance_loss(z, x, s, model, eps=1e-6):
    """
    Keep your original inertial term, but note:
    - it uses mean(acc) as a crude gravity proxy on RAW window;
      we keep it to avoid introducing extra heuristics.
    """
    B, D, T = z.shape
    C = x.shape[1]
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, : nvec * 3, :].reshape(B, nvec, 3, T)
    g = x_reshaped.mean(dim=3).mean(dim=1)  # (B,3)
    g = F.normalize(g, dim=1)

    w = s / (s.sum(dim=1, keepdim=True) + eps)
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)  # (B,D)

    z3 = model.grav_proj(z_bar)
    z3 = F.normalize(z3, dim=1)
    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# ------------------------------------------------------------------
# 6) Train / Eval
# ------------------------------------------------------------------
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for ep in range(cfg["epochs"]):
        for x, s, y in loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            z, p, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_conf, conf_ratio, thr_lo, thr_hi = confident_bce(p, y, cfg["conf_q"])
            loss_pair = pair_consistency_loss(z, s, delta=5, margin=0.2)
            loss_inert = inertial_invariance_loss(z, x, s, model)

            loss = (
                cfg["lambda_recon"] * loss_recon
                + cfg["lambda_conf"] * loss_conf
                + cfg["lambda_pair"] * loss_pair
                + cfg["lambda_inertial"] * loss_inert
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()


@torch.no_grad()
def eval_conf_bce(model, loader, conf_q: float):
    model.eval()
    losses = []
    for x, _, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        _, p, _ = model(x)
        loss, _, _, _ = confident_bce(p, y, conf_q)
        losses.append(float(loss.item()))
    return float(np.mean(losses)) if len(losses) else 0.0


# ------------------------------------------------------------------
# 7) Reporting helpers
# ------------------------------------------------------------------
def build_full_groups_from_raw(raw_24: np.ndarray, cfg):
    full_groups = {}
    for g in cfg["groups"]:
        Xg = get_group_array_from_block(raw_24, g)
        full_groups[g] = None if Xg is None else Xg.astype(np.float32)
    return full_groups


def detector_fused_on_block(raw_24: np.ndarray, cfg):
    full_groups_raw = build_full_groups_from_raw(raw_24, cfg)

    groups_dict = {}
    for g in cfg["groups"]:
        X_raw = full_groups_raw.get(g, None)
        if X_raw is None:
            continue

        X_det = maybe_smooth_for_detector(X_raw, cfg)
        c_full = compute_group_change(g, X_det, cfg)
        s_full = steady_score_from_change(c_full)

        q_full = estimate_self_consistency_quality(g, X_raw, s_full)
        groups_dict[g] = {"X_full": X_raw, "s_full": s_full, "q_full": float(q_full)}

    s_fused, w_dict, q_dict = fuse_group_scores_block(groups_dict)
    y = (1.0 - s_fused).astype(np.float32)
    y = quantile_calibrate_01(y, q=cfg["calib_q"])
    s_fused = (1.0 - y).astype(np.float32)

    mask_seg, segments, thr_hi = segmentize_from_y(y, cfg)
    return s_fused, y, mask_seg, segments, thr_hi, w_dict, q_dict


@torch.no_grad()
def infer_p_hat_on_block(model, raw_24: np.ndarray, cfg):
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    p_sum = np.zeros(T, dtype=np.float32)
    p_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)

        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        _, p, _ = model(x)
        pw = p.squeeze(0).squeeze(0).detach().cpu().numpy().astype(np.float32)

        p_sum[st:ed] += pw
        p_cnt[st:ed] += 1.0

    return p_sum / (p_cnt + 1e-8)


def compute_pair_separation_stats(model, raw_24: np.ndarray, cfg):
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    delta = 5

    _, y, _, _, _, _, _ = detector_fused_on_block(raw_24, cfg)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None

    T = len(raw_24)
    z_sum = None
    z_cnt = np.zeros(T, dtype=np.float32)

    with torch.no_grad():
        for st in range(0, T - win + 1, stride):
            ed = st + win
            Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)
            mu = Xcat.mean(axis=0, keepdims=True)
            sd = Xcat.std(axis=0, keepdims=True) + 1e-6
            Xcat = (Xcat - mu) / sd

            x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
            z, _, _ = model(x)  # (1,D,win)
            z = z.squeeze(0).detach().cpu().numpy()  # (D,win)

            if z_sum is None:
                z_sum = np.zeros((z.shape[0], T), dtype=np.float32)

            z_sum[:, st:ed] += z
            z_cnt[st:ed] += 1.0

    z_full = z_sum / (z_cnt[None, :] + 1e-8)

    if T <= delta + 1:
        return None

    z1 = z_full[:, :-delta]
    z2 = z_full[:, delta:]
    z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
    z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
    sim = np.sum(z1n * z2n, axis=0)

    y1 = y[:-delta]
    y2 = y[delta:]

    q = float(cfg.get("conf_q", 0.2))
    thr_hi = float(np.quantile(y, 1.0 - q))
    thr_lo = float(np.quantile(y, q))

    mask_ss = (y1 <= thr_lo) & (y2 <= thr_lo)
    mask_tt = (y1 >= thr_hi) & (y2 >= thr_hi)
    mask_st = (y1 <= thr_lo) & (y2 >= thr_hi)
    mask_ts = (y1 >= thr_hi) & (y2 <= thr_lo)

    return {
        "thr_lo": thr_lo,
        "thr_hi": thr_hi,
        "SS": sim[mask_ss],
        "TT": sim[mask_tt],
        "ST": sim[mask_st],
        "TS": sim[mask_ts],
    }


# ------------------------------------------------------------------
# 8) Plotting
# ------------------------------------------------------------------
def plot_raw_and_y_phat_with_segments(raw_24, y_soft, p_hat, segments, thr_hi, cfg, title, save_path):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(raw_24))
    t = np.arange(T_show) / fs

    def mag(g):
        Xg = get_group_array_from_block(raw_24[:T_show], g)
        return None if Xg is None else np.linalg.norm(Xg, axis=1)

    chest_acc = mag("chest_acc")
    ankle_acc = mag("ankle_acc")
    arm_acc = mag("arm_acc")
    ankle_gyro = mag("ankle_gyro")
    arm_gyro = mag("arm_gyro")

    plt.figure(figsize=(18, 7))
    ax1 = plt.subplot(2, 1, 1)

    for (s, e) in segments:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax1.axvspan(ss / fs, ee / fs, alpha=0.12)

    if chest_acc is not None: ax1.plot(t, chest_acc, label="chest_acc |mag|", alpha=0.9)
    if ankle_acc is not None: ax1.plot(t, ankle_acc, label="ankle_acc |mag|", alpha=0.9)
    if arm_acc is not None: ax1.plot(t, arm_acc, label="arm_acc |mag|", alpha=0.9)
    if ankle_gyro is not None: ax1.plot(t, ankle_gyro, label="ankle_gyro |mag|", alpha=0.7)
    if arm_gyro is not None: ax1.plot(t, arm_gyro, label="arm_gyro |mag|", alpha=0.7)

    ax1.set_title(title + " | raw magnitudes (shaded = confident transition segments)")
    ax1.grid(alpha=0.25)
    ax1.legend(ncol=3)

    ax2 = plt.subplot(2, 1, 2)
    for (s, e) in segments:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax2.axvspan(ss / fs, ee / fs, alpha=0.12)

    ax2.plot(t, y_soft[:T_show], label="physics y_soft (dir-change)", alpha=0.85)
    if p_hat is not None:
        ax2.plot(t, p_hat[:T_show], label="model p_hat", alpha=0.9)

    ax2.axhline(thr_hi, linestyle="--", linewidth=1.2, label=f"conf trans thr (top-q, q={cfg['seg_q']:.2f})")
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(alpha=0.25)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_fusion_weights(weights, qualities, title, save_path):
    items = sorted(weights.items(), key=lambda x: -x[1])
    labels = [k for k, _ in items]
    vals = [v for _, v in items]
    qs = [qualities.get(k, 0.0) for k in labels]

    plt.figure(figsize=(10, 4))
    x = np.arange(len(labels))
    plt.bar(x, vals)
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.title(title + " | fusion weights")
    plt.grid(axis="y", alpha=0.25)
    for i, (w, q) in enumerate(zip(vals, qs)):
        plt.text(i, w + 0.01, f"q={q:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_pair_box(pair_stats, title, save_path):
    labels = ["SS", "TT", "ST", "TS"]
    data = [pair_stats.get(k, np.array([])) for k in labels]

    plt.figure(figsize=(10, 4))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.ylim(0.0, 1.0)
    plt.title(title + " | pair separation (delta=5)")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_activity_summary(act_to_y_windows, cfg, save_path):
    acts = sorted(act_to_y_windows.keys())
    means, stds = [], []
    for act in acts:
        Ys = np.concatenate(act_to_y_windows[act], axis=0)
        means.append(float(Ys.mean()))
        stds.append(float(Ys.std()))

    x = np.arange(len(acts))
    plt.figure(figsize=(12, 5))
    plt.errorbar(x, means, yerr=stds, fmt="o", capsize=4, label="mean ± std (y_soft)")
    plt.xticks(x, [f"act{a}" for a in acts])
    plt.ylim(0, 1)
    plt.title("Activity summary of pseudo y (transition, dir-change)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_pair_sep_by_activity(pair_records, out_dir):
    acts = sorted(set([r["act"] for r in pair_records]))
    for act in acts:
        rs = [r for r in pair_records if r["act"] == act]
        if len(rs) == 0:
            continue

        SS = np.concatenate([r["SS"] for r in rs if r["SS"].size > 0], axis=0) if any(r["SS"].size > 0 for r in rs) else np.array([])
        TT = np.concatenate([r["TT"] for r in rs if r["TT"].size > 0], axis=0) if any(r["TT"].size > 0 for r in rs) else np.array([])
        ST = np.concatenate([r["ST"] for r in rs if r["ST"].size > 0], axis=0) if any(r["ST"].size > 0 for r in rs) else np.array([])
        TS = np.concatenate([r["TS"] for r in rs if r["TS"].size > 0], axis=0) if any(r["TS"].size > 0 for r in rs) else np.array([])

        data = [SS, TT, ST, TS]
        labels = ["SS", "TT", "ST", "TS"]

        plt.figure(figsize=(10, 4))
        plt.boxplot(data, labels=labels, showfliers=False)
        plt.ylim(0.0, 1.0)
        plt.grid(axis="y", alpha=0.25)
        plt.title(f"Act{act} | Pair separation (aggregated over folds)")

        meds = [np.median(d) if len(d) else np.nan for d in data]
        txt = " | ".join([f"{lab}: med={m:.3f}" for lab, m in zip(labels, meds)])
        plt.suptitle(txt, y=0.98, fontsize=9)

        save_path = os.path.join(out_dir, f"act{act}_PAIR_AGG_box.png")
        plt.tight_layout()
        plt.savefig(save_path, dpi=200)
        plt.close()
        print("[Saved]", save_path)


def pick_representative_fold(df_act: pd.DataFrame):
    if len(df_act) == 0:
        return None
    med = df_act["bce_conf"].median()
    idx = (df_act["bce_conf"] - med).abs().idxmin()
    return df_act.loc[idx].to_dict()


# ------------------------------------------------------------------
# 9) Main
# ------------------------------------------------------------------
def main():
    print("=" * 70)
    print("Main Engine (BEST v2) - Direction-change pseudo labels + Confident-only supervision")
    print("=" * 70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, CONFIG["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    pair_records = []
    act_to_y_windows = {act: [] for act in acts}
    rep_artifacts = {}

    for act in acts:
        print("\n" + "=" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("=" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = MainEngineDataset(train_blocks, CONFIG)
            test_ds = MainEngineDataset(test_blocks, CONFIG)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

            x0, _, _ = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
            ).to(DEVICE)

            train_one_fold(model, train_loader, CONFIG)
            bce_conf = eval_conf_bce(model, test_loader, CONFIG["conf_q"])

            act_to_y_windows[act].extend([d["y"] for d in train_ds.samples])

            records.append(
                {
                    "act": act,
                    "test_sub": test_sub,
                    "bce_conf": bce_conf,
                    "input_ch": input_ch,
                    "calib_q": CONFIG["calib_q"],
                    "conf_q": CONFIG["conf_q"],
                    "detector_smooth": int(CONFIG["detector_smooth"]),
                    "smooth_method": CONFIG["smooth_method"],
                    "var_win": CONFIG["var_win"],
                }
            )
            print(f"  Fold test_sub={test_sub:2d} | BCE(conf)={bce_conf:.4f}")

            # pair separation on first test block
            raw0 = test_blocks[0]["raw"]
            ps = compute_pair_separation_stats(model, raw0, CONFIG)
            if ps is not None:
                pair_records.append({"act": act, "test_sub": test_sub, "SS": ps["SS"], "TT": ps["TT"], "ST": ps["ST"], "TS": ps["TS"]})

            # save representative candidate
            rep_artifacts.setdefault(act, [])
            rep_artifacts[act].append(
                {
                    "test_sub": test_sub,
                    "bce_conf": bce_conf,
                    "raw0": raw0,
                    "subject": test_blocks[0]["subject"],
                    "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "input_ch": input_ch,
                }
            )

    df_res = pd.DataFrame(records)
    csv_path = os.path.join(CONFIG["out_dir"], "results_loso_best_v2.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # representative plots per activity
    for act in acts:
        df_act = df_res[df_res["act"] == act].reset_index(drop=True)
        if len(df_act) == 0 or act not in rep_artifacts:
            continue

        rep = pick_representative_fold(df_act)
        if rep is None:
            continue
        rep_sub = int(rep["test_sub"])

        cand = None
        for item in rep_artifacts[act]:
            if int(item["test_sub"]) == rep_sub:
                cand = item
                break
        if cand is None:
            continue

        model = MainEngineNet(
            input_ch=int(cand["input_ch"]),
            hidden_dim=CONFIG["hidden_dim"],
            latent_dim=CONFIG["latent_dim"],
        ).to(DEVICE)
        model.load_state_dict(cand["model_state"], strict=True)
        model.eval()

        raw0 = cand["raw0"]
        subj = cand["subject"]

        _, y_soft, _, segments, thr_hi, w_blk, q_blk = detector_fused_on_block(raw0, CONFIG)
        p_blk = infer_p_hat_on_block(model, raw0, CONFIG)
        pair_stats = compute_pair_separation_stats(model, raw0, CONFIG)

        p1_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_raw_ysoft_pHat_sub{subj}.png")
        plot_raw_and_y_phat_with_segments(
            raw0, y_soft, p_blk, segments, thr_hi, CONFIG,
            f"Act{act} RepFold (test_sub={rep_sub})",
            p1_path
        )
        print(f"[Saved plot] act={act} rep_sub={rep_sub} -> {p1_path}")

        p3_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_fusion_weights_sub{subj}.png")
        plot_fusion_weights(w_blk, q_blk, f"Act{act} RepFold (test_sub={rep_sub})", p3_path)

        if pair_stats is not None:
            p4_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_pair_separation_sub{subj}.png")
            plot_pair_box(pair_stats, f"Act{act} RepFold (test_sub={rep_sub})", p4_path)

    p5_path = os.path.join(CONFIG["out_dir"], "P5_activity_summary.png")
    plot_activity_summary(act_to_y_windows, CONFIG, p5_path)
    print("[Saved]", p5_path)

    # aggregated pair separation per activity
    if len(pair_records) > 0:
        plot_pair_sep_by_activity(pair_records, CONFIG["out_dir"])

    print("\nDone.")


if __name__ == "__main__":
    main()


Main Engine (BEST v2) - Direction-change pseudo labels + Confident-only supervision
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | BCE(conf)=1.2325
  Fold test_sub= 2 | BCE(conf)=1.8634
  Fold test_sub= 3 | BCE(conf)=0.8250
  Fold test_sub= 4 | BCE(conf)=1.2494
  Fold test_sub= 5 | BCE(conf)=0.7444
  Fold test_sub= 6 | BCE(conf)=1.0405
  Fold test_sub= 7 | BCE(conf)=0.6656
  Fold test_sub= 8 | BCE(conf)=0.9598
  Fold test_sub= 9 | BCE(conf)=1.6868
  Fold test_sub=10 | BCE(conf)=1.6995

[Activity 7] Single-activity LOSO
[Blocks] act=7 total=10 | folds=10
  Fold test_sub= 1 | BCE(conf)=0.1293
  Fold test_sub= 2 | BCE(conf)=0.2646
  Fold test_sub= 3 | BCE(conf)=0.2390
  Fold test_sub= 4 | BCE(conf)=0.1133
  Fold test_sub= 5 | BC

/tmp/ipython-input-1479734544.py:906: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plot] act=7 rep_sub=2 -> ./out_main_engine_best_v2/act7_Rep_raw_ysoft_pHat_sub2.png


/tmp/ipython-input-1479734544.py:906: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plot] act=12 rep_sub=3 -> ./out_main_engine_best_v2/act12_Rep_raw_ysoft_pHat_sub3.png


...

[Saved] ./out_main_engine_best_v2/act6_PAIR_AGG_box.png


/tmp/ipython-input-1479734544.py:952: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved] ./out_main_engine_best_v2/act7_PAIR_AGG_box.png


/tmp/ipython-input-1479734544.py:952: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved] ./out_main_engine_best_v2/act12_PAIR_AGG_box.png

Done.


In [3]:
"""
Main Engine (RESET ver.)
- Physics-defined transition indicator is NOT a supervision target.
- We learn representations by:
  (1) Steady-weighted latent flow stability
  (2) Steady/Transition pair separation (SS/TT/ST/TS) via contrast-style loss
  (3) Steady-weighted inertial invariance
  (4) Optional reconstruction

- Transition detection/analysis is done post-hoc using:
  - latent flow magnitude m(t) = ||z(t+Δ)-z(t)|| (or 1-cos sim)
  - and cross-checked/overlaid with physics indicator τ(t)

Outputs:
- per-activity LOSO metrics CSV
- representative plots per activity:
  P1: raw magnitudes + physics τ(t) + model flow m(t) + physics segments shading
  P2: fusion weights
  P3: SS/TT/ST/TS similarity boxplot
  P4: SS/TT/ST/TS scatter (τ_pair vs sim) (optional)
  P5: activity summary of τ(t)
"""

import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter
import matplotlib.pyplot as plt


# =============================================================================
# 0) Utils
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def robust_zscore(x: np.ndarray, eps=1e-6, mad_floor=1e-3):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    mad = max(float(mad), float(mad_floor))
    return (x - med) / mad


def cosine_sim_np(a: np.ndarray, b: np.ndarray, eps=1e-8):
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    return float(np.dot(a, b) / (na * nb))


# =============================================================================
# 1) MHEALTH Loading
# =============================================================================
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        try:
            df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        except:
            df = pd.read_csv(fpath, sep="\t", header=None)

        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count (expected >=24).")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id
        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(
        f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}"
    )
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    # MHEALTH columns: (0:2 chest_acc), ...
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    """
    Each (subject, activity) forms one block (continuous segment in that label for that subject file).
    Note: this treats all samples of that label for that subject as one long segment (as in your code).
    """
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# =============================================================================
# 2) Physics-defined indicator (detector) + fusion
# =============================================================================
def ema_smooth(x: np.ndarray, alpha: float):
    if x.ndim == 1:
        y = np.zeros_like(x, dtype=np.float32)
        y[0] = x[0]
        for t in range(1, len(x)):
            y[t] = alpha * x[t] + (1 - alpha) * y[t - 1]
        return y.astype(np.float32)
    else:
        y = np.zeros_like(x, dtype=np.float32)
        y[0, :] = x[0, :]
        for t in range(1, x.shape[0]):
            y[t, :] = alpha * x[t, :] + (1 - alpha) * y[t - 1, :]
        return y.astype(np.float32)


def savgol_smooth(x: np.ndarray, win: int, poly: int):
    win = int(win)
    if win < 3:
        return x.astype(np.float32)
    if win % 2 == 0:
        win += 1
    if x.ndim == 1:
        if len(x) < win:
            return x.astype(np.float32)
        return savgol_filter(x, window_length=win, polyorder=poly).astype(np.float32)
    else:
        T, C = x.shape
        if T < win:
            return x.astype(np.float32)
        y = np.zeros_like(x, dtype=np.float32)
        for c in range(C):
            y[:, c] = savgol_filter(x[:, c], window_length=win, polyorder=poly).astype(np.float32)
        return y


def maybe_smooth_for_detector(Xg: np.ndarray, cfg):
    if Xg is None:
        return None
    if not cfg.get("detector_smooth", False):
        return Xg.astype(np.float32)

    method = cfg.get("smooth_method", "savgol")
    if method == "ema":
        return ema_smooth(Xg.astype(np.float32), alpha=float(cfg.get("ema_alpha", 0.15)))
    return savgol_smooth(
        Xg.astype(np.float32),
        win=int(cfg.get("savgol_win", 11)),
        poly=int(cfg.get("savgol_poly", 2)),
    )


def acc_change_score(acc_3: np.ndarray, var_win: int = 10):
    # direction change + direction variance
    T = len(acc_3)
    u = acc_3 / (np.linalg.norm(acc_3, axis=1, keepdims=True) + 1e-8)
    cos_prev = np.sum(u[1:] * u[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)

    dir_change = np.zeros(T, dtype=np.float32)
    dir_change[1:] = 1.0 - cos_prev

    half = var_win // 2
    dir_var = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = u[s:e]
        dir_var[i] = float(np.mean(np.var(local, axis=0)))

    return (dir_change + dir_var).astype(np.float32)


def gyro_change_score(gyro_3: np.ndarray, var_win: int = 10):
    # magnitude change + local variance
    T = len(gyro_3)
    mag = np.linalg.norm(gyro_3, axis=1)

    dmag = np.zeros(T, dtype=np.float32)
    dmag[1:] = np.abs(mag[1:] - mag[:-1])

    half = var_win // 2
    lvar = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        lvar[i] = float(np.var(mag[s:e]))

    return (dmag + lvar).astype(np.float32)


def compute_group_change(group_key: str, Xg: np.ndarray):
    if Xg is None:
        return None
    if "acc" in group_key:
        return acc_change_score(Xg, var_win=10)
    if "gyro" in group_key:
        return gyro_change_score(Xg, var_win=10)
    return np.linalg.norm(np.diff(Xg, axis=0, prepend=Xg[:1]), axis=1).astype(np.float32)


def transition_indicator_from_change(c: np.ndarray):
    """
    Physics-defined indicator:
      - large c => more transition-like
    returns:
      tau(t) in [0,1]
      s(t)=1-tau in [0,1]
    """
    cz = robust_zscore(c, eps=1e-6, mad_floor=1e-3)
    b = np.percentile(cz, 80)
    tau = sigmoid_np((cz - b) / 0.5).astype(np.float32)
    if len(tau) >= 11:
        tau = savgol_filter(tau, window_length=11, polyorder=2).astype(np.float32)
        tau = np.clip(tau, 0.0, 1.0)
    s = (1.0 - tau).astype(np.float32)
    return tau, s


def estimate_self_consistency_quality(group_key: str, X_full: np.ndarray, s_full: np.ndarray):
    """
    quality: similarity(high-steady) - similarity(low-steady)
    """
    q = 0.2
    k_pairs = 128

    T = len(s_full)
    order = np.argsort(s_full)
    n = max(5, int(q * T))
    low_idx = order[:n]
    high_idx = order[-n:]
    if len(low_idx) < 5 or len(high_idx) < 5:
        return 0.0

    if "acc" in group_key:
        V = X_full / (np.linalg.norm(X_full, axis=1, keepdims=True) + 1e-8)
    else:
        V = X_full

    def mean_pair_sim(idxs):
        sims = []
        for _ in range(k_pairs):
            i, j = np.random.choice(idxs, size=2, replace=True)
            sims.append(cosine_sim_np(V[i], V[j]))
        return float(np.mean(sims))

    S_high = mean_pair_sim(high_idx)
    S_low = mean_pair_sim(low_idx)
    return float(S_high - S_low)


def fuse_group_scores_block(groups_dict):
    avail = [(g, d) for g, d in groups_dict.items() if d["tau_full"] is not None]
    if not avail:
        raise ValueError("No available groups for fusion.")

    qs = np.array([d["q_full"] for _, d in avail], dtype=np.float32)
    tau_softmax = float(groups_dict.get("_tau_softmax", 0.25))  # not used; keep compatibility
    tau_temp = 0.25  # fixed
    ws = np.exp(qs / max(tau_temp, 1e-6))
    ws = ws / (ws.sum() + 1e-8)

    T = len(avail[0][1]["tau_full"])
    tau_fused = np.zeros(T, dtype=np.float32)
    weights, qualities = {}, {}
    for (g, d), w in zip(avail, ws):
        tau_fused += float(w) * d["tau_full"]
        weights[g] = float(w)
        qualities[g] = float(d["q_full"])

    tau_fused = np.clip(tau_fused, 0.0, 1.0)
    s_fused = (1.0 - tau_fused).astype(np.float32)
    return tau_fused, s_fused, weights, qualities


def quantile_calibrate_01(y: np.ndarray, q: float, eps: float = 1e-6):
    """
    calibrate to [0,1] based on quantiles (robust scaling).
    """
    q = float(np.clip(q, 0.0, 0.49))
    lo = float(np.quantile(y, q))
    hi = float(np.quantile(y, 1.0 - q))
    denom = max(hi - lo, eps)
    y_cal = (y - lo) / denom
    return np.clip(y_cal, 0.0, 1.0).astype(np.float32)


def mask_fill_gaps(mask: np.ndarray, max_gap: int):
    if max_gap <= 0:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if m[i]:
            i += 1
            continue
        j = i
        while j < T and (not m[j]):
            j += 1
        gap_len = j - i
        if i > 0 and j < T and gap_len <= max_gap:
            m[i:j] = True
        i = j
    return m


def mask_remove_short_islands(mask: np.ndarray, min_len: int):
    if min_len <= 1:
        return mask
    m = mask.astype(bool).copy()
    T = len(m)
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        seg_len = j - i
        if seg_len < min_len:
            m[i:j] = False
        i = j
    return m


def segmentize_from_tau(tau01: np.ndarray, cfg):
    """
    tau01: (T,) in [0,1], higher = more transition-like.
    returns: mask, segments(list of (s,e)), thr_hi
    """
    q = float(cfg.get("seg_q", 0.2))
    thr_hi = float(np.quantile(tau01, 1.0 - q))

    mask = (tau01 >= thr_hi)

    fs = int(cfg["fs"])
    min_len = int(round(float(cfg.get("seg_min_len_sec", 0.2)) * fs))
    max_gap = int(round(float(cfg.get("seg_max_gap_sec", 0.1)) * fs))

    mask = mask_fill_gaps(mask, max_gap=max_gap)
    mask = mask_remove_short_islands(mask, min_len=min_len)

    m = mask.astype(bool)
    T = len(m)
    segments = []
    i = 0
    while i < T:
        if not m[i]:
            i += 1
            continue
        j = i
        while j < T and m[j]:
            j += 1
        segments.append((i, j))
        i = j

    return m, segments, thr_hi


def build_full_groups_from_raw(raw_24: np.ndarray, cfg):
    full_groups = {}
    for g in cfg["groups"]:
        Xg = get_group_array_from_block(raw_24, g)
        full_groups[g] = None if Xg is None else Xg.astype(np.float32)
    return full_groups


def physics_fused_on_block(raw_24: np.ndarray, cfg):
    """
    Returns:
      tau_fused, s_fused in [0,1]
      segments for visualization
      weights/qualities per group
    """
    full_groups_raw = build_full_groups_from_raw(raw_24, cfg)
    groups_dict = {}

    for g in cfg["groups"]:
        X_raw = full_groups_raw.get(g, None)
        if X_raw is None:
            continue

        X_det = maybe_smooth_for_detector(X_raw, cfg)
        c_full = compute_group_change(g, X_det)
        tau_full, s_full = transition_indicator_from_change(c_full)

        # quality computed on RAW to avoid "cheating"
        q_full = estimate_self_consistency_quality(g, X_raw, s_full)
        groups_dict[g] = {"X_full": X_raw, "tau_full": tau_full, "s_full": s_full, "q_full": float(q_full)}

    tau_fused, s_fused, w_dict, q_dict = fuse_group_scores_block(groups_dict)

    # robust calibration
    tau_fused = quantile_calibrate_01(tau_fused, q=float(cfg.get("calib_q", 0.2)))
    s_fused = (1.0 - tau_fused).astype(np.float32)

    mask_seg, segments, thr_hi = segmentize_from_tau(tau_fused, cfg)
    return tau_fused, s_fused, mask_seg, segments, thr_hi, w_dict, q_dict


# =============================================================================
# 3) Dataset
# =============================================================================
class MainEngineDataset(Dataset):
    """
    Returns:
      x: (C,T) raw (normalized per-window)  [MODEL INPUT]
      s: (T,) steady prior in [0,1]         [PHYSICS PRIOR, NOT LABEL]
      tau: (T,) transition indicator in [0,1]
    """
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]
        self.samples = []

        for b in blocks:
            raw = b["raw"]
            subject = b["subject"]
            act = b["act"]
            T = len(raw)

            # raw groups for MODEL INPUT (no smoothing)
            full_groups_raw = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups_raw[g] = None if Xg is None else Xg.astype(np.float32)

            # physics on full block
            tau_full, s_full, _, _, _, w_block, q_block = physics_fused_on_block(raw, cfg)

            avail_groups = [g for g in self.groups if full_groups_raw.get(g, None) is not None]
            if len(avail_groups) == 0:
                continue

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win

                x_parts = [full_groups_raw[g][st:ed] for g in avail_groups]
                if len(x_parts) == 0:
                    continue
                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)

                # per-window normalization
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append(
                    {
                        "x": Xcat,
                        "s": s_full[st:ed].astype(np.float32),
                        "tau": tau_full[st:ed].astype(np.float32),
                        "meta": {"subject": subject, "act": act, "weights": w_block, "qualities": q_block},
                    }
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s"], dtype=torch.float32)                  # (T,)
        tau = torch.tensor(d["tau"], dtype=torch.float32)              # (T,)
        return x, s, tau


# =============================================================================
# 4) Model (clean, only what we need)
# =============================================================================
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )
        # For inertial invariance head
        self.grav_proj = nn.Linear(latent_dim, 3)

        # Optional reconstruction decoder (kept to stabilize training)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.to_latent(h)        # (B,D,T)
        x_recon = self.decoder(z)    # (B,C,T)
        return z, x_recon


# =============================================================================
# 5) Losses (NO BCE to physics indicator)
# =============================================================================
def steady_flow_loss(z, s, delta: int = 1, eps=1e-6):
    """
    Encourage small latent change in steady regions:
      L = E[ w(t) * (1 - cos(z_t, z_{t+δ})) ]
    where w(t) = s(t) * s(t+δ)
    """
    B, D, T = z.shape
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s[:, :-delta]
    s2 = s[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    cos = (z1n * z2n).sum(dim=1)  # (B,T-d)
    flow = (1.0 - cos)            # higher = more change

    w = (s1 * s2).detach()
    w = w / (w.mean() + eps)      # normalize weight scale
    return (w * flow).mean()


def pair_separation_loss(z, s, delta: int = 5, margin: float = 0.2, eps=1e-6):
    """
    SS/TT separation using physics prior only as weights:
      - If both steady (high s): want sim high  -> (1 - sim)
      - If both transition (low s): want sim low -> relu(sim - margin)
    """
    B, D, T = z.shape
    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]
    z2 = z[:, :, delta:]
    s1 = s[:, :-delta]
    s2 = s[:, delta:]

    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)  # (B,T-d)

    # weights (detach) - do not backprop to physics prior
    w_ss = (s1 * s2).detach()
    w_tt = ((1.0 - s1) * (1.0 - s2)).detach()

    # normalize to keep stable scale
    w_ss = w_ss / (w_ss.mean() + eps)
    w_tt = w_tt / (w_tt.mean() + eps)

    loss_ss = (w_ss * (1.0 - sim)).mean()
    loss_tt = (w_tt * F.relu(sim - margin)).mean()
    return loss_ss + loss_tt


def inertial_invariance_loss(z, x, s, model, eps=1e-6):
    """
    In steady regions, latent should not encode gravity direction strongly.
    Implementation:
      g = mean of all 3D vectors across time and vector-groups from x (approx gravity direction)
      z_bar = steady-weighted temporal average of z
      model.grav_proj(z_bar) should be weakly correlated with g.
    We minimize |cos| correlation (but you can also maximize orthogonality).
    """
    B, D, T = z.shape
    C = x.shape[1]
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, : nvec * 3, :].reshape(B, nvec, 3, T)
    g = x_reshaped.mean(dim=3).mean(dim=1)  # (B,3)
    g = F.normalize(g, dim=1)

    # steady-weighted average z
    w = s / (s.sum(dim=1, keepdim=True) + eps)  # (B,T)
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)     # (B,D)

    z3 = model.grav_proj(z_bar)
    z3 = F.normalize(z3, dim=1)

    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# =============================================================================
# 6) Train / Eval (post-hoc metrics)
# =============================================================================
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for _ in range(cfg["epochs"]):
        for x, s, tau in loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            tau = tau.to(DEVICE)  # not directly used in losses now

            z, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_flow = steady_flow_loss(z, s, delta=int(cfg.get("flow_delta", 1)))
            loss_pair = pair_separation_loss(z, s, delta=int(cfg.get("pair_delta", 5)), margin=float(cfg.get("pair_margin", 0.2)))
            loss_inert = inertial_invariance_loss(z, x, s, model)

            loss = (
                cfg["lambda_recon"] * loss_recon
                + cfg["lambda_flow"] * loss_flow
                + cfg["lambda_pair"] * loss_pair
                + cfg["lambda_inertial"] * loss_inert
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        sched.step()


@torch.no_grad()
def infer_latent_flow_on_block(model, raw_24: np.ndarray, cfg):
    """
    Build full-length z(t) (by overlap-averaging window z),
    then compute flow magnitude m(t)=1-cos(z_t,z_{t+δ}).
    """
    model.eval()
    win = cfg["window_size"]
    stride = cfg["stride"]
    T = len(raw_24)

    full_groups = build_full_groups_from_raw(raw_24, cfg)
    avail_groups = [g for g in cfg["groups"] if full_groups.get(g, None) is not None]
    if len(avail_groups) == 0:
        return None, None

    z_sum = None
    z_cnt = np.zeros(T, dtype=np.float32)

    for st in range(0, T - win + 1, stride):
        ed = st + win
        Xcat = np.concatenate([full_groups[g][st:ed] for g in avail_groups], axis=1).astype(np.float32)
        mu = Xcat.mean(axis=0, keepdims=True)
        sd = Xcat.std(axis=0, keepdims=True) + 1e-6
        Xcat = (Xcat - mu) / sd

        x = torch.tensor(Xcat, dtype=torch.float32).transpose(0, 1).unsqueeze(0).to(DEVICE)
        z, _ = model(x)                 # (1,D,win)
        z = z.squeeze(0).detach().cpu().numpy().astype(np.float32)  # (D,win)

        if z_sum is None:
            z_sum = np.zeros((z.shape[0], T), dtype=np.float32)

        z_sum[:, st:ed] += z
        z_cnt[st:ed] += 1.0

    z_full = z_sum / (z_cnt[None, :] + 1e-8)  # (D,T)

    delta = int(cfg.get("flow_delta_eval", 5))
    if T <= delta:
        return z_full, None

    z1 = z_full[:, :-delta]
    z2 = z_full[:, delta:]
    z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
    z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
    cos = np.sum(z1n * z2n, axis=0)     # (T-d)
    m = (1.0 - cos).astype(np.float32)  # higher => more change
    # pad to length T
    m_full = np.zeros(T, dtype=np.float32)
    m_full[delta:] = m
    return z_full, m_full


def auc_binary(scores: np.ndarray, labels01: np.ndarray):
    """
    Simple AUC without sklearn:
    labels01 in {0,1}
    """
    scores = scores.astype(np.float64)
    labels01 = labels01.astype(np.int32)
    pos = scores[labels01 == 1]
    neg = scores[labels01 == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    # probability that pos score > neg score (with tie=0.5)
    cnt = 0.0
    for p in pos:
        cnt += np.sum(p > neg) + 0.5 * np.sum(p == neg)
    return float(cnt / (len(pos) * len(neg)))


@torch.no_grad()
def eval_fold_metrics_on_blocks(model, test_blocks, cfg):
    """
    Post-hoc evaluation WITHOUT GT transition:
    - corr( m(t), tau(t) )
    - AUC: discriminate top-q tau vs bottom-q tau using m(t)
    - SS/TT/ST/TS similarity distributions on z
    """
    qs = float(cfg.get("eval_q", 0.2))
    delta = int(cfg.get("flow_delta_eval", 5))

    corrs = []
    aucs = []
    ss_all, tt_all, st_all, ts_all = [], [], [], []

    for b in test_blocks:
        raw = b["raw"]
        tau, s, _, _, _, _, _ = physics_fused_on_block(raw, cfg)
        z_full, m = infer_latent_flow_on_block(model, raw, cfg)
        if m is None or z_full is None:
            continue

        # corr between m and tau
        if np.std(m) > 1e-8 and np.std(tau) > 1e-8:
            corrs.append(float(np.corrcoef(m, tau)[0, 1]))

        # AUC using top-q as positives and bottom-q as negatives (exclude mid)
        thr_hi = float(np.quantile(tau, 1.0 - qs))
        thr_lo = float(np.quantile(tau, qs))
        y_bin = np.full_like(tau, -1, dtype=np.int32)
        y_bin[tau >= thr_hi] = 1
        y_bin[tau <= thr_lo] = 0
        mask = (y_bin >= 0)
        if np.sum(y_bin[mask] == 1) > 10 and np.sum(y_bin[mask] == 0) > 10:
            aucs.append(auc_binary(m[mask], y_bin[mask]))

        # SS/TT/ST/TS similarity on z using tau thresholds
        # sim(t)=cos(z_t, z_{t+δ})
        T = z_full.shape[1]
        if T <= delta + 1:
            continue

        z1 = z_full[:, :-delta]
        z2 = z_full[:, delta:]
        z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
        z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
        sim = np.sum(z1n * z2n, axis=0)

        tau1 = tau[:-delta]
        tau2 = tau[delta:]
        thr_hi = float(np.quantile(tau, 1.0 - qs))
        thr_lo = float(np.quantile(tau, qs))

        SS = sim[(tau1 <= thr_lo) & (tau2 <= thr_lo)]
        TT = sim[(tau1 >= thr_hi) & (tau2 >= thr_hi)]
        ST = sim[(tau1 <= thr_lo) & (tau2 >= thr_hi)]
        TS = sim[(tau1 >= thr_hi) & (tau2 <= thr_lo)]

        if len(SS) > 0: ss_all.append(SS)
        if len(TT) > 0: tt_all.append(TT)
        if len(ST) > 0: st_all.append(ST)
        if len(TS) > 0: ts_all.append(TS)

    def cat_or_empty(L):
        return np.concatenate(L, axis=0) if len(L) else np.array([], dtype=np.float32)

    out = {
        "corr_m_tau": float(np.nanmean(corrs)) if len(corrs) else np.nan,
        "auc_m_tau": float(np.nanmean(aucs)) if len(aucs) else np.nan,
        "SS": cat_or_empty(ss_all),
        "TT": cat_or_empty(tt_all),
        "ST": cat_or_empty(st_all),
        "TS": cat_or_empty(ts_all),
    }
    return out


# =============================================================================
# 7) Plotting
# =============================================================================
def plot_raw_tau_flow_with_segments(raw_24, tau, m_flow, segments, thr_hi, cfg, title, save_path):
    fs = cfg["fs"]
    T_show = min(int(cfg["plot_sec"] * fs), len(raw_24))
    t = np.arange(T_show) / fs

    def mag(g):
        Xg = get_group_array_from_block(raw_24[:T_show], g)
        return None if Xg is None else np.linalg.norm(Xg, axis=1)

    chest_acc = mag("chest_acc")
    ankle_acc = mag("ankle_acc")
    arm_acc = mag("arm_acc")
    ankle_gyro = mag("ankle_gyro")
    arm_gyro = mag("arm_gyro")

    plt.figure(figsize=(18, 7))
    ax1 = plt.subplot(2, 1, 1)

    for (s, e) in segments:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax1.axvspan(ss / fs, ee / fs, alpha=0.12)

    if chest_acc is not None: ax1.plot(t, chest_acc, label="chest_acc |mag|", alpha=0.9)
    if ankle_acc is not None: ax1.plot(t, ankle_acc, label="ankle_acc |mag|", alpha=0.9)
    if arm_acc is not None: ax1.plot(t, arm_acc, label="arm_acc |mag|", alpha=0.9)
    if ankle_gyro is not None: ax1.plot(t, ankle_gyro, label="ankle_gyro |mag|", alpha=0.7)
    if arm_gyro is not None: ax1.plot(t, arm_gyro, label="arm_gyro |mag|", alpha=0.7)

    ax1.set_title(title + " | raw magnitudes (shaded=physics top-q segments)")
    ax1.grid(alpha=0.25)
    ax1.legend(ncol=3)

    ax2 = plt.subplot(2, 1, 2)
    for (s, e) in segments:
        if s >= T_show:
            continue
        ss = max(0, s)
        ee = min(T_show, e)
        ax2.axvspan(ss / fs, ee / fs, alpha=0.12)

    ax2.plot(t, tau[:T_show], label="physics τ(t) (transition indicator)", alpha=0.9)
    ax2.axhline(thr_hi, linestyle="--", linewidth=1.2, label=f"τ thr_hi (top-q, q={cfg['seg_q']:.2f})")

    if m_flow is not None:
        # normalize m_flow to [0,1] for overlay
        mf = m_flow[:T_show].copy()
        if np.std(mf) > 1e-8:
            mf = (mf - mf.min()) / (mf.max() - mf.min() + 1e-8)
        ax2.plot(t, mf, label="model flow m(t) (normalized)", alpha=0.9)

    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(alpha=0.25)
    ax2.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_fusion_weights(weights, qualities, title, save_path):
    items = sorted(weights.items(), key=lambda x: -x[1])
    labels = [k for k, _ in items]
    vals = [v for _, v in items]
    qs = [qualities.get(k, 0.0) for k in labels]

    plt.figure(figsize=(10, 4))
    x = np.arange(len(labels))
    plt.bar(x, vals)
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylim(0, 1)
    plt.title(title + " | fusion weights")
    plt.grid(axis="y", alpha=0.25)
    for i, (w, q) in enumerate(zip(vals, qs)):
        plt.text(i, w + 0.01, f"q={q:.2f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_pair_box(pair_stats, title, save_path):
    labels = ["SS", "TT", "ST", "TS"]
    data = [pair_stats.get(k, np.array([])) for k in labels]

    plt.figure(figsize=(10, 4))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(title + " | SS/TT/ST/TS similarity (cos)")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_pair_scatter_tau_sim(raw_24, z_full, tau, cfg, title, save_path):
    """
    Scatter of (tau_pair, sim) for t and t+delta.
    This helps you visually inspect separation beyond boxplots.
    """
    delta = int(cfg.get("flow_delta_eval", 5))
    T = z_full.shape[1]
    if T <= delta + 1:
        return

    z1 = z_full[:, :-delta]
    z2 = z_full[:, delta:]
    z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
    z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
    sim = np.sum(z1n * z2n, axis=0)

    tau1 = tau[:-delta]
    tau2 = tau[delta:]
    tau_pair = 0.5 * (tau1 + tau2)

    # downsample for readability
    step = max(1, len(sim) // 4000)
    sim_ds = sim[::step]
    tau_ds = tau_pair[::step]

    plt.figure(figsize=(7, 5))
    plt.scatter(tau_ds, sim_ds, s=6, alpha=0.35)
    plt.xlabel("tau_pair (mean of τ(t), τ(t+δ))")
    plt.ylabel("cos sim(z(t), z(t+δ))")
    plt.title(title + " | tau vs similarity scatter")
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_activity_summary_tau(act_to_tau_windows, cfg, save_path):
    acts = sorted(act_to_tau_windows.keys())
    means, stds = [], []
    for act in acts:
        Ts = np.concatenate(act_to_tau_windows[act], axis=0)
        means.append(float(Ts.mean()))
        stds.append(float(Ts.std()))

    x = np.arange(len(acts))
    plt.figure(figsize=(12, 5))
    plt.errorbar(x, means, yerr=stds, fmt="o", capsize=4, label="mean ± std (physics τ)")
    plt.xticks(x, [f"act{a}" for a in acts])
    plt.ylim(0, 1)
    plt.title("Activity summary of physics τ(t) (transition indicator)")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def pick_representative_fold(df_act: pd.DataFrame, key="corr_m_tau"):
    """
    Pick a representative fold around median (robust).
    If corr is nan, fallback to first.
    """
    if len(df_act) == 0:
        return None
    if df_act[key].notna().sum() == 0:
        return df_act.iloc[0].to_dict()
    med = df_act[key].median()
    idx = (df_act[key] - med).abs().idxmin()
    return df_act.loc[idx].to_dict()


# =============================================================================
# 8) Main (CONFIG inside main as requested)
# =============================================================================
def main():
    CONFIG = {
        "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
        "target_activities": [6, 7, 12],
        "fs": 50,

        "window_size": 100,
        "stride": 50,
        "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

        "batch_size": 64,
        "epochs": 50,
        "lr": 1e-3,
        "seed": 42,

        "latent_dim": 64,
        "hidden_dim": 128,

        # -------- training losses (RESET) --------
        "lambda_recon": 1.0,
        "lambda_flow": 0.7,       # steady-weighted flow stability
        "lambda_pair": 0.4,       # SS/TT separation
        "lambda_inertial": 0.2,   # steady-weighted inertial invariance

        # -------- physics indicator calibration --------
        "calib_q": 0.2,

        # detector-only smoothing (model input stays raw-normalized)
        "detector_smooth": True,
        "smooth_method": "savgol",  # "savgol" or "ema"
        "savgol_win": 11,
        "savgol_poly": 2,
        "ema_alpha": 0.15,

        # segmentize for visualization (NOT hysteresis)
        "seg_q": 0.2,
        "seg_min_len_sec": 0.20,
        "seg_max_gap_sec": 0.10,

        # flow deltas
        "flow_delta": 1,          # training flow loss uses δ=1
        "pair_delta": 5,          # pair separation uses δ=5
        "pair_margin": 0.2,       # TT similarity upper margin
        "flow_delta_eval": 5,     # eval/plot flow uses δ=5

        # eval thresholds
        "eval_q": 0.2,

        "out_dir": "./out_main_engine_reset",
        "plot_sec": 60,
    }

    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    set_seed(CONFIG["seed"])

    print("=" * 70)
    print("Main Engine (RESET) - physics prior (NOT target) + representation learning")
    print("=" * 70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, CONFIG["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    act_to_tau_windows = {act: [] for act in acts}
    rep_artifacts = {}

    for act in acts:
        print("\n" + "=" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("=" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = MainEngineDataset(train_blocks, CONFIG)
            test_ds = MainEngineDataset(test_blocks, CONFIG)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False)

            # collect physics tau stats (from train windows)
            act_to_tau_windows[act].extend([d["tau"] for d in train_ds.samples])

            x0, _, _ = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
            ).to(DEVICE)

            train_one_fold(model, train_loader, CONFIG)

            # post-hoc eval on test blocks (no GT transition)
            metrics = eval_fold_metrics_on_blocks(model, test_blocks, CONFIG)

            records.append(
                {
                    "act": act,
                    "test_sub": test_sub,
                    "input_ch": input_ch,
                    "corr_m_tau": metrics["corr_m_tau"],
                    "auc_m_tau": metrics["auc_m_tau"],
                    "detector_smooth": int(CONFIG["detector_smooth"]),
                    "smooth_method": CONFIG["smooth_method"],
                    "calib_q": CONFIG["calib_q"],
                    "seg_q": CONFIG["seg_q"],
                }
            )

            print(
                f"  Fold test_sub={test_sub:2d} | corr(m,τ)={metrics['corr_m_tau']:.4f} | AUC(m->τ top/bot)={metrics['auc_m_tau']:.4f}"
            )

            # store representative artifact candidate
            b0 = test_blocks[0]
            rep_artifacts.setdefault(act, [])
            rep_artifacts[act].append(
                {
                    "test_sub": test_sub,
                    "corr_m_tau": metrics["corr_m_tau"],
                    "raw0": b0["raw"],
                    "subject": b0["subject"],
                    "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                    "input_ch": input_ch,
                }
            )

    df_res = pd.DataFrame(records)
    csv_path = os.path.join(CONFIG["out_dir"], "results_loso_reset.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # representative plots per activity
    for act in acts:
        df_act = df_res[df_res["act"] == act].reset_index(drop=True)
        if len(df_act) == 0 or act not in rep_artifacts:
            continue

        rep = pick_representative_fold(df_act, key="corr_m_tau")
        if rep is None:
            continue
        rep_sub = int(rep["test_sub"])

        cand = None
        for item in rep_artifacts[act]:
            if int(item["test_sub"]) == rep_sub:
                cand = item
                break
        if cand is None:
            continue

        model = MainEngineNet(
            input_ch=int(cand["input_ch"]),
            hidden_dim=CONFIG["hidden_dim"],
            latent_dim=CONFIG["latent_dim"],
        ).to(DEVICE)
        model.load_state_dict(cand["model_state"], strict=True)
        model.eval()

        raw0 = cand["raw0"]
        subj = cand["subject"]

        # physics prior and segments
        tau, s, _, segments, thr_hi, w_blk, q_blk = physics_fused_on_block(raw0, CONFIG)

        # model latent and flow
        z_full, m_flow = infer_latent_flow_on_block(model, raw0, CONFIG)

        # compute SS/TT/ST/TS arrays for plot
        qs = float(CONFIG.get("eval_q", 0.2))
        delta = int(CONFIG.get("flow_delta_eval", 5))
        pair_stats = {"SS": np.array([]), "TT": np.array([]), "ST": np.array([]), "TS": np.array([])}
        if z_full is not None and z_full.shape[1] > delta + 1:
            T = z_full.shape[1]
            z1 = z_full[:, :-delta]
            z2 = z_full[:, delta:]
            z1n = z1 / (np.linalg.norm(z1, axis=0, keepdims=True) + 1e-8)
            z2n = z2 / (np.linalg.norm(z2, axis=0, keepdims=True) + 1e-8)
            sim = np.sum(z1n * z2n, axis=0)

            tau1 = tau[:-delta]
            tau2 = tau[delta:]
            thr_hi2 = float(np.quantile(tau, 1.0 - qs))
            thr_lo2 = float(np.quantile(tau, qs))
            pair_stats["SS"] = sim[(tau1 <= thr_lo2) & (tau2 <= thr_lo2)]
            pair_stats["TT"] = sim[(tau1 >= thr_hi2) & (tau2 >= thr_hi2)]
            pair_stats["ST"] = sim[(tau1 <= thr_lo2) & (tau2 >= thr_hi2)]
            pair_stats["TS"] = sim[(tau1 >= thr_hi2) & (tau2 <= thr_lo2)]

        p1_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_raw_tau_flow_sub{subj}.png")
        plot_raw_tau_flow_with_segments(
            raw0, tau, m_flow, segments, thr_hi, CONFIG,
            f"Act{act} RepFold (test_sub={rep_sub})",
            p1_path
        )
        print(f"[Saved plot] act={act} rep_sub={rep_sub} -> {p1_path}")

        p2_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_fusion_weights_sub{subj}.png")
        plot_fusion_weights(w_blk, q_blk, f"Act{act} RepFold (test_sub={rep_sub})", p2_path)

        p3_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_pair_box_sub{subj}.png")
        plot_pair_box(pair_stats, f"Act{act} RepFold (test_sub={rep_sub})", p3_path)

        if z_full is not None:
            p4_path = os.path.join(CONFIG["out_dir"], f"act{act}_Rep_tau_sim_scatter_sub{subj}.png")
            plot_pair_scatter_tau_sim(raw0, z_full, tau, CONFIG, f"Act{act} RepFold (test_sub={rep_sub})", p4_path)

    p5_path = os.path.join(CONFIG["out_dir"], "P5_activity_summary_tau.png")
    plot_activity_summary_tau(act_to_tau_windows, CONFIG, p5_path)
    print("[Saved]", p5_path)

    print("\nDone.")


if __name__ == "__main__":
    main()


Main Engine (RESET) - physics prior (NOT target) + representation learning
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | corr(m,τ)=0.4097 | AUC(m->τ top/bot)=0.8087
  Fold test_sub= 2 | corr(m,τ)=0.3816 | AUC(m->τ top/bot)=0.8238
  Fold test_sub= 3 | corr(m,τ)=0.3674 | AUC(m->τ top/bot)=0.7866
  Fold test_sub= 4 | corr(m,τ)=0.3266 | AUC(m->τ top/bot)=0.8050
  Fold test_sub= 5 | corr(m,τ)=0.4395 | AUC(m->τ top/bot)=0.8525
  Fold test_sub= 6 | corr(m,τ)=0.4659 | AUC(m->τ top/bot)=0.8881
  Fold test_sub= 7 | corr(m,τ)=0.2621 | AUC(m->τ top/bot)=0.7545
  Fold test_sub= 8 | corr(m,τ)=0.4196 | AUC(m->τ top/bot)=0.8446
  Fold test_sub= 9 | corr(m,τ)=0.3696 | AUC(m->τ top/bot)=0.7872
  Fold test_sub=10 | corr(m,τ)=0.4198 | AUC(m->τ 

/tmp/ipython-input-716036218.py:917: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plot] act=7 rep_sub=3 -> ./out_main_engine_reset/act7_Rep_raw_tau_flow_sub3.png


/tmp/ipython-input-716036218.py:917: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved plot] act=12 rep_sub=1 -> ./out_main_engine_reset/act12_Rep_raw_tau_flow_sub1.png


/tmp/ipython-input-716036218.py:917: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data, labels=labels, showfliers=False)


[Saved] ./out_main_engine_reset/P5_activity_summary_tau.png

Done.


In [1]:
import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt


# ============================================================
# 0) Common utils
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def robust_norm01(x: np.ndarray, q=0.02, eps=1e-6):
    """Robustly map to [0,1] per-trial (no act-specific)."""
    lo = float(np.quantile(x, q))
    hi = float(np.quantile(x, 1.0 - q))
    denom = max(hi - lo, eps)
    y = (x - lo) / denom
    return np.clip(y, 0.0, 1.0).astype(np.float32)


def safe_norm_rows(X: np.ndarray, eps=1e-8):
    n = np.linalg.norm(X, axis=1, keepdims=True) + eps
    return X / n


def ema1d(x: np.ndarray, alpha: float):
    y = np.zeros_like(x, dtype=np.float32)
    y[0] = x[0]
    for t in range(1, len(x)):
        y[t] = alpha * x[t] + (1 - alpha) * y[t - 1]
    return y


# ============================================================
# 1) MHEALTH loading + block building
# ============================================================
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count.")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id
        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}")
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    # MHEALTH columns (common mapping used in your code)
    if group == "chest_acc":   return block_np[:, 0:3]
    if group == "ankle_acc":   return block_np[:, 5:8]
    if group == "arm_acc":     return block_np[:, 14:17]
    if group == "ankle_gyro":  return block_np[:, 8:11]
    if group == "arm_gyro":    return block_np[:, 17:20]
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame, win_size: int):
    blocks = []
    base_cols = list(range(24))
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= win_size:
                blocks.append({"subject": int(sub), "act": int(act), "raw": raw})
    return blocks


# ============================================================
# 2) Prior bank (Teacher): fixed physics scores from raw
#    - No act-specific choice
#    - We'll compute K priors per sample: phi_k(t) in [0,1]
# ============================================================
def acc_mag(acc3: np.ndarray):
    return np.linalg.norm(acc3, axis=1).astype(np.float32)


def gyro_mag(g3: np.ndarray):
    return np.linalg.norm(g3, axis=1).astype(np.float32)


def jerk_mag(acc3: np.ndarray, fs: int):
    # jerk = derivative of acc
    d = np.diff(acc3, axis=0, prepend=acc3[:1])
    j = d * float(fs)
    return np.linalg.norm(j, axis=1).astype(np.float32)


def dir_change(acc3: np.ndarray):
    # 1 - cos(u_t, u_{t-1}) (gravity direction proxy; uses normalized vectors)
    u = safe_norm_rows(acc3)
    cos_prev = np.sum(u[1:] * u[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)
    out = np.zeros(len(acc3), dtype=np.float32)
    out[1:] = (1.0 - cos_prev).astype(np.float32)
    return out


def compute_prior_bank(raw24: np.ndarray, cfg):
    """
    Returns:
      phi: (T, K) in [0,1]
      names: list[str] length K
    """
    fs = int(cfg["fs"])
    # choose a fixed bank: 4 priors, but each prior is multi-sensor pooled (no act-specific)
    # 1) Motion quietness: -acc_mag pooled
    # 2) Rotation quietness: -gyro_mag pooled
    # 3) Jerk quietness: -jerk_mag pooled
    # 4) Reorientation: dir_change pooled
    groups = cfg["groups"]

    acc_mags, gyro_mags, jerk_mags, dir_changes = [], [], [], []
    for g in groups:
        X = get_group_array_from_block(raw24, g)
        if X is None:
            continue
        if "acc" in g:
            acc_mags.append(acc_mag(X))
            jerk_mags.append(jerk_mag(X, fs=fs))
            dir_changes.append(dir_change(X))
        if "gyro" in g:
            gyro_mags.append(gyro_mag(X))

    if len(acc_mags) == 0 or len(gyro_mags) == 0:
        raise ValueError("Not enough sensor groups to compute priors.")

    # pool across available sensors by mean (still fixed, not per-act)
    acc_pool = np.mean(np.stack(acc_mags, axis=1), axis=1)
    gyro_pool = np.mean(np.stack(gyro_mags, axis=1), axis=1)
    jerk_pool = np.mean(np.stack(jerk_mags, axis=1), axis=1) if len(jerk_mags) > 0 else np.zeros(len(raw24), np.float32)
    dir_pool  = np.mean(np.stack(dir_changes, axis=1), axis=1) if len(dir_changes) > 0 else np.zeros(len(raw24), np.float32)

    # IMPORTANT:
    # boundary often corresponds to "valley/quiet" for many acts → use inverted magnitudes
    # But we keep also reorientation prior.
    # Normalize each per-trial to [0,1] robustly (NOT act-specific).
    q_acc  = robust_norm01(-acc_pool, q=cfg["prior_norm_q"])
    q_gyro = robust_norm01(-gyro_pool, q=cfg["prior_norm_q"])
    q_jerk = robust_norm01(-jerk_pool, q=cfg["prior_norm_q"])
    r_dir  = robust_norm01(+dir_pool,  q=cfg["prior_norm_q"])

    # optional mild smoothing for teacher τ to reduce flicker (still physics side)
    if cfg.get("teacher_ema_alpha", 0.0) > 0:
        a = float(cfg["teacher_ema_alpha"])
        q_acc  = ema1d(q_acc,  a)
        q_gyro = ema1d(q_gyro, a)
        q_jerk = ema1d(q_jerk, a)
        r_dir  = ema1d(r_dir,  a)

    phi = np.stack([q_acc, q_gyro, q_jerk, r_dir], axis=1).astype(np.float32)
    names = ["quiet_acc", "quiet_gyro", "quiet_jerk", "reorient_dir"]
    return phi, names


def make_teacher_tau(phi: np.ndarray, cfg):
    """
    Teacher τ_prior(t) in [0,1] from phi(t) via fixed rule.
    Rule must be ACT-INDEPENDENT and not tuned per act.
    We use a soft "OR" style aggregation:
      τ = 1 - Π_k (1 - phi_k)  (any prior high => τ high)
    Then calibrate to [0,1] robustly per-trial.
    """
    one_minus = 1.0 - np.clip(phi, 0.0, 1.0)
    prod = np.prod(one_minus, axis=1)
    tau = 1.0 - prod
    tau = tau.astype(np.float32)
    tau = robust_norm01(tau, q=cfg["tau_norm_q"])
    return tau


# ============================================================
# 3) Dataset: raw windows + teacher τ windows (distillation target)
#    JSON NOT USED.
# ============================================================
class DistillDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]
        self.samples = []

        for b in blocks:
            raw = b["raw"]
            T = len(raw)

            # teacher: prior bank -> tau_prior (full length)
            phi_full, _ = compute_prior_bank(raw, cfg)
            tau_full = make_teacher_tau(phi_full, cfg)  # (T,)

            # model input = concatenated raw groups (standardized per-window)
            full_groups = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                full_groups[g] = None if Xg is None else Xg.astype(np.float32)

            avail = [g for g in self.groups if full_groups.get(g, None) is not None]
            if len(avail) == 0:
                continue

            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win
                x_parts = [full_groups[g][st:ed] for g in avail]
                Xcat = np.concatenate(x_parts, axis=1).astype(np.float32)  # (win, C)

                # per-window standardize (no act-specific)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                tau_w = tau_full[st:ed].astype(np.float32)

                self.samples.append({
                    "x": Xcat,          # (win,C)
                    "tau": tau_w,       # (win,)
                    "meta": {"act": b["act"], "sub": b["subject"]}
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        tau = torch.tensor(d["tau"], dtype=torch.float32)              # (T,)
        return x, tau


# ============================================================
# 4) Student model: raw -> gate(t) -> tau_hat(t)
#    gate chooses weights over prior bank (K=4 fixed)
# ============================================================
class GatedPriorNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, K: int):
        super().__init__()
        self.K = K
        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        # gate logits per time: (B,K,T)
        self.gate_head = nn.Conv1d(hidden_dim, K, kernel_size=1)
        # boundary logit per time from gated priors (scalar temperature)
        self.temp = nn.Parameter(torch.tensor(1.0))

    def forward(self, x, phi_student):
        """
        x: (B,C,T) raw
        phi_student: (B,K,T) student's estimate of priors?  -> 여기서는 teacher priors를 넣지 않음.
        BUT 우리는 "raw만 입력"을 원하므로, phi_student를 모델 내부에서 직접 만들 수도 있음.
        지금 버전에서는 더 깔끔하게:
          - gate는 raw에서 만들고
          - priors(φ)는 teacher 정의 그대로 raw로부터 '고정 계산'해 데이터로더에서 같이 줄 수도 있음.
        다만 유저 요구가 'raw만'이므로, φ도 raw로부터 on-the-fly 계산하는 블록을 모델에 넣는다.
        """
        h = self.backbone(x)
        logits = self.gate_head(h)            # (B,K,T)
        w = torch.softmax(logits, dim=1)      # (B,K,T)
        # mixture
        tau_hat = (w * phi_student).sum(dim=1)  # (B,T)
        # sharpen/scale
        tau_hat = torch.sigmoid(self.temp * (tau_hat * 2 - 1))  # map [0,1] -> logit-ish then sigmoid
        return tau_hat, w


# ---- φ computation inside PyTorch (so raw only) ----
def torch_safe_norm_rows(X, eps=1e-8):
    n = torch.linalg.norm(X, dim=1, keepdim=True) + eps
    return X / n

def torch_robust_norm01(x, q=0.02, eps=1e-6):
    """
    x: (B,T) tensor
    per-sample robust scaling to [0,1] using quantiles along T.
    """
    B, T = x.shape
    k_lo = max(0, int(round(q * (T - 1))))
    k_hi = min(T - 1, int(round((1.0 - q) * (T - 1))))
    x_sorted, _ = torch.sort(x, dim=1)
    lo = x_sorted[:, k_lo]
    hi = x_sorted[:, k_hi]
    denom = torch.clamp(hi - lo, min=eps)
    y = (x - lo[:, None]) / denom[:, None]
    return torch.clamp(y, 0.0, 1.0)

def compute_phi_torch_from_raw(x, cfg):
    """
    x: (B,C,T) where C is concatenated [chest_acc(3), ankle_acc(3), arm_acc(3), ankle_gyro(3), arm_gyro(3)] etc
    BUT since groups can vary, we assume you keep groups order fixed like cfg["groups"] and always concat in that order.
    To avoid mismatch, we will enforce group list exactly as in cfg and require those exist.
    """
    B, C, T = x.shape
    fs = float(cfg["fs"])

    # parse channel slices based on cfg["groups"] order and each group has 3ch
    # (this matches your previous concatenation behavior)
    group_names = cfg["groups"]
    slices = {}
    ch = 0
    for g in group_names:
        slices[g] = (ch, ch + 3)
        ch += 3
    if ch != C:
        # if some groups missing, this torch version isn't safe.
        # keep it strict to avoid silent bugs.
        raise ValueError(f"Channel mismatch: expected {ch} from groups*3, got {C}. Ensure all groups exist and are concatenated in fixed order.")

    acc_list = []
    gyro_list = []
    jerk_list = []
    dir_list = []

    for g in group_names:
        s, e = slices[g]
        Xg = x[:, s:e, :]  # (B,3,T)
        if "acc" in g:
            acc_m = torch.linalg.norm(Xg, dim=1)                # (B,T)
            acc_list.append(acc_m)
            # jerk
            d = torch.diff(Xg, dim=2, prepend=Xg[:, :, :1]) * fs
            jerk_m = torch.linalg.norm(d, dim=1)                # (B,T)
            jerk_list.append(jerk_m)
            # dir change
            u = torch_safe_norm_rows(Xg.permute(0,2,1).reshape(-1,3)).reshape(B,T,3).permute(0,2,1)  # (B,3,T)
            # cos between u_t and u_{t-1}
            cos_prev = (u[:, :, 1:] * u[:, :, :-1]).sum(dim=1).clamp(-1.0, 1.0)  # (B,T-1)
            dc = torch.zeros((B,T), device=x.device, dtype=x.dtype)
            dc[:, 1:] = 1.0 - cos_prev
            dir_list.append(dc)
        if "gyro" in g:
            gyro_m = torch.linalg.norm(Xg, dim=1)               # (B,T)
            gyro_list.append(gyro_m)

    acc_pool  = torch.stack(acc_list, dim=1).mean(dim=1)        # (B,T)
    gyro_pool = torch.stack(gyro_list, dim=1).mean(dim=1)       # (B,T)
    jerk_pool = torch.stack(jerk_list, dim=1).mean(dim=1) if len(jerk_list) > 0 else torch.zeros_like(acc_pool)
    dir_pool  = torch.stack(dir_list, dim=1).mean(dim=1)  if len(dir_list)  > 0 else torch.zeros_like(acc_pool)

    # quiet priors: invert magnitudes
    q = float(cfg["prior_norm_q"])
    quiet_acc  = torch_robust_norm01(-acc_pool,  q=q)
    quiet_gyro = torch_robust_norm01(-gyro_pool, q=q)
    quiet_jerk = torch_robust_norm01(-jerk_pool, q=q)
    reorient   = torch_robust_norm01(+dir_pool,  q=q)

    phi = torch.stack([quiet_acc, quiet_gyro, quiet_jerk, reorient], dim=1)  # (B,K,T)
    return phi


def teacher_tau_from_phi_torch(phi, cfg):
    # τ = 1 - Π (1-phi)
    one_minus = 1.0 - torch.clamp(phi, 0.0, 1.0)
    prod = torch.prod(one_minus, dim=1)  # (B,T)
    tau = 1.0 - prod
    tau = torch_robust_norm01(tau, q=float(cfg["tau_norm_q"]))
    return tau


# ============================================================
# 5) Loss: distillation + optional regularizers
# ============================================================
def distill_loss(tau_hat, tau_teacher, w, cfg):
    """
    tau_hat: (B,T) student
    tau_teacher: (B,T) teacher
    w: (B,K,T) gate weights
    """
    # main
    loss = F.mse_loss(tau_hat, tau_teacher)

    # gate entropy regularizer (avoid degenerate uniform gates OR overly spiky early)
    lam_ent = float(cfg["lambda_gate_entropy"])
    if lam_ent > 0:
        ent = -(w * (w.clamp_min(1e-8).log())).sum(dim=1).mean()  # higher = more uniform
        # we want moderate entropy: push away from extremes by penalizing too-low entropy
        loss = loss + lam_ent * F.relu(float(cfg["entropy_floor"]) - ent)

    # temporal smoothness of tau_hat (reduce flicker)
    lam_tv = float(cfg["lambda_tau_smooth"])
    if lam_tv > 0:
        tv = torch.mean(torch.abs(tau_hat[:, 1:] - tau_hat[:, :-1]))
        loss = loss + lam_tv * tv

    # sparsity prior (boundaries shouldn't dominate all time)
    lam_sparse = float(cfg["lambda_tau_sparse"])
    if lam_sparse > 0:
        target_mean = float(cfg["tau_mean_target"])
        m = tau_hat.mean()
        loss = loss + lam_sparse * (m - target_mean) ** 2

    return loss


# ============================================================
# 6) Train / Eval helpers
# ============================================================
def train_one_fold(model, loader, cfg, device):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    model.train()

    for ep in range(cfg["epochs"]):
        for x, _tau_teacher_placeholder in loader:
            x = x.to(device)  # (B,C,T)

            # teacher τ computed from raw inside training (still label-free)
            phi = compute_phi_torch_from_raw(x, cfg)                 # (B,K,T)
            tau_teacher = teacher_tau_from_phi_torch(phi, cfg)       # (B,T)

            tau_hat, w = model(x, phi.detach())  # detach phi: gate learns mapping raw->weights, not phi itself
            loss = distill_loss(tau_hat, tau_teacher, w, cfg)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()


@torch.no_grad()
def eval_corr_auc_like(model, loader, cfg, device):
    """
    Since no GT, report:
      - corr(tau_hat, tau_teacher)
      - AUC-like: distinguish teacher top-q vs bottom-q by tau_hat
    """
    model.eval()
    all_hat, all_tch = [], []
    for x, _ in loader:
        x = x.to(device)
        phi = compute_phi_torch_from_raw(x, cfg)
        tau_t = teacher_tau_from_phi_torch(phi, cfg)
        tau_hat, _w = model(x, phi)
        all_hat.append(tau_hat.cpu().numpy())
        all_tch.append(tau_t.cpu().numpy())

    H = np.concatenate(all_hat, axis=0).reshape(-1)
    T = np.concatenate(all_tch, axis=0).reshape(-1)

    # corr
    corr = float(np.corrcoef(H, T)[0, 1])

    # AUC-like: top/bottom quantiles of teacher as pseudo-classes (for sanity only)
    q = float(cfg["eval_topbot_q"])
    thr_hi = float(np.quantile(T, 1.0 - q))
    thr_lo = float(np.quantile(T, q))
    pos = H[T >= thr_hi]
    neg = H[T <= thr_lo]
    if len(pos) == 0 or len(neg) == 0:
        auc_like = float("nan")
    else:
        # probability that random pos > random neg
        # (fast approx)
        m = min(2000, len(pos), len(neg))
        pos_s = np.random.choice(pos, size=m, replace=False)
        neg_s = np.random.choice(neg, size=m, replace=False)
        auc_like = float((pos_s[:, None] > neg_s[None, :]).mean())

    return corr, auc_like


# ============================================================
# 7) Plot: show raw + tau_teacher + tau_hat + gate weights
# ============================================================
@torch.no_grad()
def infer_full_on_block(model, raw24, cfg, device):
    """
    Slide windows and overlap-add tau_hat (like your p_hat aggregator).
    Also compute tau_teacher directly on full sequence for reference.
    """
    fs = int(cfg["fs"])
    win = int(cfg["window_size"])
    stride = int(cfg["stride"])

    # build fixed-order concatenation for all groups (strict)
    parts = []
    for g in cfg["groups"]:
        X = get_group_array_from_block(raw24, g)
        if X is None:
            raise ValueError(f"Missing group {g} in block for strict torch phi computation.")
        parts.append(X.astype(np.float32))
    Xfull = np.concatenate(parts, axis=1)  # (T, C)
    Tlen, C = Xfull.shape

    # teacher full
    # (numpy version)
    phi_np, _names = compute_prior_bank(raw24, cfg)
    tau_teacher_full = make_teacher_tau(phi_np, cfg)  # (T,)

    # overlap-add student
    tau_sum = np.zeros(Tlen, np.float32)
    tau_cnt = np.zeros(Tlen, np.float32)

    # gate weight average (K)
    K = 4
    w_sum = np.zeros((K, Tlen), np.float32)
    w_cnt = np.zeros(Tlen, np.float32)

    for st in range(0, Tlen - win + 1, stride):
        ed = st + win
        xw = Xfull[st:ed].copy()
        mu = xw.mean(axis=0, keepdims=True)
        sd = xw.std(axis=0, keepdims=True) + 1e-6
        xw = (xw - mu) / sd
        xt = torch.tensor(xw, dtype=torch.float32).transpose(0,1).unsqueeze(0).to(device)  # (1,C,win)

        phi_t = compute_phi_torch_from_raw(xt, cfg)  # (1,K,win)
        tau_hat, w = model(xt, phi_t)

        tau_hat = tau_hat.squeeze(0).cpu().numpy().astype(np.float32)
        w = w.squeeze(0).cpu().numpy().astype(np.float32)  # (K,win)

        tau_sum[st:ed] += tau_hat
        tau_cnt[st:ed] += 1.0
        w_sum[:, st:ed] += w
        w_cnt[st:ed] += 1.0

    tau_hat_full = tau_sum / (tau_cnt + 1e-8)
    w_full = w_sum / (w_cnt[None, :] + 1e-8)

    return tau_teacher_full, tau_hat_full, w_full


def plot_rep_trial(raw24, tau_t, tau_h, w_full, cfg, title, save_path):
    fs = int(cfg["fs"])
    T_show = min(int(cfg["plot_sec"] * fs), len(raw24))
    t = np.arange(T_show) / fs

    # choose a view signal just for plotting (not act-specific for logic; only for visualization)
    view = cfg["plot_view_signal"]
    if view == "chest_acc_mag":
        X = get_group_array_from_block(raw24[:T_show], "chest_acc")
        y = np.linalg.norm(X, axis=1)
        ylab = "chest_acc |mag|"
    elif view == "arm_gyro_mag":
        X = get_group_array_from_block(raw24[:T_show], "arm_gyro")
        y = np.linalg.norm(X, axis=1)
        ylab = "arm_gyro |mag|"
    else:
        X = get_group_array_from_block(raw24[:T_show], "arm_acc")
        y = np.linalg.norm(X, axis=1)
        ylab = "arm_acc |mag|"

    plt.figure(figsize=(18, 9))
    ax1 = plt.subplot(3,1,1)
    ax1.plot(t, y, alpha=0.9, label=ylab)
    ax1.set_title(title + " | raw view")
    ax1.grid(alpha=0.25)
    ax1.legend()

    ax2 = plt.subplot(3,1,2)
    ax2.plot(t, tau_t[:T_show], label="tau_teacher(prior)", alpha=0.9)
    ax2.plot(t, tau_h[:T_show], label="tau_hat(student)", alpha=0.9)
    ax2.set_ylim(-0.05, 1.05)
    ax2.set_title("Teacher vs Student boundary score")
    ax2.grid(alpha=0.25)
    ax2.legend()

    ax3 = plt.subplot(3,1,3)
    names = ["quiet_acc","quiet_gyro","quiet_jerk","reorient_dir"]
    for k in range(4):
        ax3.plot(t, w_full[k, :T_show], label=f"gate w[{names[k]}]", alpha=0.9)
    ax3.set_ylim(-0.05, 1.05)
    ax3.set_title("Gate weights over prior bank (data-driven, act-agnostic)")
    ax3.grid(alpha=0.25)
    ax3.legend(ncol=2)

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


# ============================================================
# 8) Main (LOSO per-activity optional, but training is label-free)
# ============================================================
def main():
    CONFIG = {
        "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
        "target_activities": [6, 7, 12],
        "fs": 50,

        # fixed groups (STRICT order; required for torch phi parsing)
        "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

        # windowing
        "window_size": 100,
        "stride": 50,

        # training
        "batch_size": 64,
        "epochs": 30,
        "lr": 1e-3,
        "hidden_dim": 128,
        "seed": 42,

        # teacher normalization / smoothing (act-agnostic)
        "prior_norm_q": 0.02,
        "tau_norm_q": 0.05,
        "teacher_ema_alpha": 0.0,   # 0.05~0.15 주면 teacher flicker 줄어듦 (원하면 켜)

        # regularizers
        "lambda_gate_entropy": 0.02,
        "entropy_floor": 0.6,       # soft floor on entropy
        "lambda_tau_smooth": 0.02,  # TV
        "lambda_tau_sparse": 0.10,  # mean target
        "tau_mean_target": 0.20,    # boundary 비율 prior (너 seg_cov 참고해서 조절 가능)

        # evaluation sanity (still label-free)
        "eval_topbot_q": 0.2,

        # plotting
        "plot_sec": 60,
        "plot_view_signal": "chest_acc_mag",

        # outputs
        "out_dir": "./out_prior_gate_distill",
    }

    os.makedirs(CONFIG["out_dir"], exist_ok=True)
    set_seed(CONFIG["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    print("="*70)
    print("Prior-bank Boundary (Teacher) + Raw-driven Gating (Student) [NO JSON TRAIN]")
    print("="*70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all, CONFIG["window_size"])
    acts = sorted({b["act"] for b in blocks_all})

    records = []
    rep_artifacts = {}

    for act in acts:
        print("\n" + "="*70)
        print(f"[Activity {act}] Single-activity LOSO (label-free distill)")
        print("="*70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        for test_sub in subjects:
            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks  = [b for b in blocks if b["subject"] == test_sub]
            if len(train_blocks) == 0 or len(test_blocks) == 0:
                continue

            train_ds = DistillDataset(train_blocks, CONFIG)
            test_ds  = DistillDataset(test_blocks,  CONFIG)
            if len(train_ds) == 0 or len(test_ds) == 0:
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False)

            # infer input_ch strictly from groups*3
            input_ch = len(CONFIG["groups"]) * 3
            K = 4
            model = GatedPriorNet(input_ch=input_ch, hidden_dim=CONFIG["hidden_dim"], K=K).to(device)

            train_one_fold(model, train_loader, CONFIG, device)
            corr, auc_like = eval_corr_auc_like(model, test_loader, CONFIG, device)

            records.append({"act": act, "test_sub": test_sub, "corr": corr, "auc_like": auc_like})
            print(f"  Fold test_sub={test_sub:2d} | corr(tau_hat, tau_tch)={corr:.4f} | AUC_like(top/bot)={auc_like:.4f}")

            # keep one representative artifact per act: first fold
            rep_artifacts.setdefault(act, None)
            if rep_artifacts[act] is None:
                rep_artifacts[act] = {
                    "test_sub": test_sub,
                    "raw0": test_blocks[0]["raw"],
                    "subject": test_blocks[0]["subject"],
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                }

    # save results
    df_res = pd.DataFrame(records)
    csv_path = os.path.join(CONFIG["out_dir"], "results_loso_distill.csv")
    df_res.to_csv(csv_path, index=False)
    print("\n[Saved]", csv_path)

    # representative plots
    for act, item in rep_artifacts.items():
        if item is None:
            continue
        model = GatedPriorNet(input_ch=len(CONFIG["groups"])*3, hidden_dim=CONFIG["hidden_dim"], K=4).to(device)
        model.load_state_dict(item["state"], strict=True)
        model.eval()

        raw0 = item["raw0"]
        tau_t, tau_h, w_full = infer_full_on_block(model, raw0, CONFIG, device)
        save_path = os.path.join(CONFIG["out_dir"], f"act{act}_rep_sub{item['subject']}_teacher_student_gate.png")
        plot_rep_trial(raw0, tau_t, tau_h, w_full, CONFIG, f"Act{act} RepFold(test_sub={item['test_sub']})", save_path)
        print("[Saved plot]", save_path)

    print("\nDone.")


if __name__ == "__main__":
    main()


Device: cuda
Prior-bank Boundary (Teacher) + Raw-driven Gating (Student) [NO JSON TRAIN]
[Data] Total samples: 68098 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(12)]

[Activity 6] Single-activity LOSO (label-free distill)
[Blocks] act=6 total=10 | folds=10
  Fold test_sub= 1 | corr(tau_hat, tau_tch)=0.8578 | AUC_like(top/bot)=0.9878
  Fold test_sub= 2 | corr(tau_hat, tau_tch)=0.8459 | AUC_like(top/bot)=0.9877
  Fold test_sub= 3 | corr(tau_hat, tau_tch)=0.8369 | AUC_like(top/bot)=0.9821
  Fold test_sub= 4 | corr(tau_hat, tau_tch)=0.8270 | AUC_like(top/bot)=0.9754
  Fold test_sub= 5 | corr(tau_hat, tau_tch)=0.8353 | AUC_like(top/bot)=0.9805
  Fold test_sub= 6 | corr(tau_hat, tau_tch)=0.8908 | AUC_like(top/bot)=0.9950
  Fold test_sub= 7 | corr(tau_hat, tau_tch)=0.8599 | AUC_like(top/bot)=0.9934
  Fold test_sub= 8 | corr(tau_hat, tau_tch)=0.7849 | 